# Preseleccionador de candidatos para Movimientos en Masa usando NDVI, pendiente y mapa de susceptibilidad 

## 1 - Preparando librerias y autenticación con GEE

In [ ]:
import sys

# Instala librerías faltantes en el entorno activo de miniconda
packages = [
    "earthengine-api",
    "geopandas",
    "rasterio",
    "requests",
    "Pillow",
    "scipy",
    "numpy",
    "matplotlib",
]

import subprocess
for pkg in packages:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", pkg],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"✓ {pkg}")
    else:
        print(f"✗ {pkg}: {result.stderr.strip()}")

print("\nInstalación completada. Reinicia el kernel si es la primera vez.")

In [ ]:
import ee

# Autenticación para entorno local (abre navegador la primera vez)
# Si ya autenticaste antes, puedes comentar la línea ee.Authenticate()
try:
    ee.Initialize(project='ornate-woodland-468014-i6')
    print("✓ Earth Engine inicializado correctamente (credenciales existentes)")
except Exception:
    print("Iniciando autenticación...")
    ee.Authenticate()  # Abre navegador para autenticar
    ee.Initialize(project='ornate-woodland-468014-i6')
    print("✓ Earth Engine autenticado e inicializado")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
import rasterio
from rasterio.transform import from_bounds
from rasterio.crs import CRS
import requests
from io import BytesIO
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

print("✓ Librerías importadas correctamente")

## 4b - Configuración de umbrales (percentiles)


In [ ]:
# ============================================================
# CONFIGURACIÓN CENTRAL DE UMBRALES
# Cambiar SOLO aquí. Todo el pipeline aguas abajo lo hereda.
# ============================================================

PCT_NDVI = 95          # percentil de ΔNDVI para los candidatos iniciales
PCT_BSI  = 25          # percentil de ΔBSI (calculado sobre los candidatos NDVI)

# Segundo umbral de ΔNDVI aplicado DESPUÉS de todos los filtros
# (es el que alimenta el DBSCAN / detección de objetos).
# Poner 0 para desactivarlo (los objetos se generan sobre mask_final completo).
PCT_NDVI_FINAL = PCT_NDVI

# Etiquetas usadas en títulos y leyendas de todas las figuras
LBL_NDVI       = f"P{PCT_NDVI}"
LBL_BSI        = f"P{PCT_BSI}"
LBL_NDVI_FINAL = f"P{PCT_NDVI_FINAL}"

print("Configuración de umbrales:")
print(f"  ΔNDVI  → percentil {PCT_NDVI}")
print(f"  ΔBSI   → percentil {PCT_BSI} (sobre candidatos NDVI)")
print(f"  ΔNDVI final (objetos) → percentil {PCT_NDVI_FINAL}"
      + ("  [desactivado]" if PCT_NDVI_FINAL <= 0 else ""))


## 2 - Definiendo funciones

In [ ]:
import math
import zipfile
import requests
import numpy as np
from io import BytesIO
from PIL import Image
from shapely.geometry import box
from rasterio.merge import merge
from rasterio.io import MemoryFile
import warnings
warnings.filterwarnings('ignore')

# ─── 1. Conversión Shapely → GEE ────────────────────────────────────────────
def shapely_to_ee(geom):
    if geom.geom_type == "Polygon":
        exterior = [[x, y] for x, y, *_ in geom.exterior.coords]
        if exterior[0] != exterior[-1]:
            exterior.append(exterior[0])
        rings = [exterior]
        for interior in geom.interiors:
            coords_2d = [[x, y] for x, y, *_ in interior.coords]
            if coords_2d[0] != coords_2d[-1]:
                coords_2d.append(coords_2d[0])
            rings.append(coords_2d)
        return ee.Geometry.Polygon(rings)
    elif geom.geom_type == "MultiPolygon":
        multipoly = []
        for poly in geom.geoms:
            exterior = [[x, y] for x, y, *_ in poly.exterior.coords]
            if exterior[0] != exterior[-1]:
                exterior.append(exterior[0])
            rings = [exterior]
            for interior in poly.interiors:
                coords_2d = [[x, y] for x, y, *_ in interior.coords]
                if coords_2d[0] != coords_2d[-1]:
                    coords_2d.append(coords_2d[0])
                rings.append(coords_2d)
            multipoly.append(rings)
        return ee.Geometry.MultiPolygon(multipoly)
    else:
        raise ValueError(f"Geometría no soportada: {geom.geom_type}")


# ─── 2. Máscara de nubes ─────────────────────────────────────────────────────
def mask_clouds(image):
    scl        = image.select('SCL')
    cloud_prob = image.select('MSK_CLDPRB')
    ndsi       = image.normalizedDifference(['B3', 'B11'])
    cloud_mask = (scl.neq(3).And(scl.neq(8)).And(scl.neq(9))
                  .And(cloud_prob.lt(5)).And(ndsi.lt(0.8)))
    cloud_mask = cloud_mask.focal_min(radius=2).focal_max(radius=2)
    return image.updateMask(cloud_mask).select(['B2', 'B3', 'B4', 'B8', 'B11'])


# ─── 3. NDVI ─────────────────────────────────────────────────────────────────
def calculate_ndvi(image):
    return image.addBands(image.normalizedDifference(['B8', 'B4']).rename('NDVI'))


# ─── 4. Creación de mosaico ──────────────────────────────────────────────────
def create_mosaic(start_date, end_date, aoi):
    collection = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
                  .filterBounds(aoi)
                  .filterDate(start_date, end_date)
                  .map(mask_clouds))
    count = collection.size().getInfo()
    print(f"  → Imágenes encontradas ({start_date} a {end_date}): {count}")

    collection_ndvi = collection.map(lambda img:
        img.addBands(img.normalizedDifference(['B8', 'B4']).rename('NDVI_QA')))
    mosaic = collection_ndvi.qualityMosaic('NDVI_QA')

    # Máscara post-mosaico de remanentes de nubes
    ndvi       = mosaic.normalizedDifference(['B8', 'B4'])
    bright     = mosaic.select('B2').gt(1500)
    b2_b8_ratio = mosaic.select('B2').divide(mosaic.select('B8').add(1))
    cloud_ratio = b2_b8_ratio.gt(1.5)
    cloud_remnant = ndvi.lt(-0.05)
    final_cloud_mask = cloud_remnant.Or(bright).Or(cloud_ratio)
    mosaic = mosaic.updateMask(final_cloud_mask.Not())

    mosaic = calculate_ndvi(mosaic)
    return mosaic


# ─── 5. Grilla de tiles ──────────────────────────────────────────────────────
def make_tile_grid(geom, tile_size_deg=0.15):
    minx, miny, maxx, maxy = geom.bounds
    tiles = []
    x = minx
    while x < maxx:
        y = miny
        while y < maxy:
            tile    = box(x, y, min(x + tile_size_deg, maxx), min(y + tile_size_deg, maxy))
            clipped = geom.intersection(tile)
            if not clipped.is_empty:
                tiles.append(clipped)
            y += tile_size_deg
        x += tile_size_deg
    return tiles


# ─── 6. Descarga de un tile ──────────────────────────────────────────────────
def download_tile(image, tile_geom, scale, bands, is_ndvi=False, max_retries=3, timeout=180):
    import time

    ee_tile = shapely_to_ee(tile_geom)
    params  = {'region': ee_tile, 'scale': scale, 'format': 'GEO_TIFF', 'crs': 'EPSG:4326'}
    sel     = ['NDVI'] if is_ndvi else bands
    url     = image.select(sel).getDownloadURL(params)

    for attempt in range(1, max_retries + 1):
        try:
            resp    = requests.get(url, timeout=timeout)
            content = resp.content

            if zipfile.is_zipfile(BytesIO(content)):
                with zipfile.ZipFile(BytesIO(content)) as zf:
                    tif_name = [n for n in zf.namelist() if n.endswith('.tif')][0]
                    return zf.read(tif_name)
            elif b'"error"' in content[:200]:
                print(f"    ✗ Error GEE: {content.decode()[:100]}")
                return None
            else:
                return content  # TIF directo sin zip

        except requests.exceptions.Timeout:
            print(f"    ⏱ Timeout (intento {attempt}/{max_retries})...", flush=True)
            time.sleep(10 * attempt)
        except Exception as e:
            print(f"    ✗ {e} (intento {attempt}/{max_retries})")
            time.sleep(10 * attempt)

    print(f"    ✗ Falló después de {max_retries} intentos")
    return None


def download_mosaic_tiled(image, geom, scale, bands, is_ndvi=False):
    import time

    tiles = make_tile_grid(geom, tile_size_deg=0.05)  # tamaño validado
    print(f"    Grilla: {len(tiles)} tiles a {scale}m/px  (~{len(tiles)*15//60}h {len(tiles)*15%60}min estimado)")

    memfiles, datasets, failed = [], [], 0

    for i, tile_geom in enumerate(tiles):
        print(f"    Tile {i+1}/{len(tiles)}...", end=' ', flush=True)
        tif_bytes = download_tile(image, tile_geom, scale, bands, is_ndvi=is_ndvi)

        if tif_bytes is None:
            print("omitido"); failed += 1; continue

        try:
            mf = MemoryFile(tif_bytes)
            ds = mf.open()
            memfiles.append(mf)
            datasets.append(ds)
            print(f"✓ {ds.width}×{ds.height}px")
        except Exception as e:
            print(f"✗ rasterio: {e}"); failed += 1

        # Pausa pequeña para no saturar la API
        time.sleep(1)

    if not datasets:
        raise RuntimeError("Todos los tiles fallaron.")

    print(f"\n    Ensamblando {len(datasets)} tiles ({failed} fallidos)...")
    merged, transform = merge(datasets)

    for ds in datasets: ds.close()
    for mf in memfiles: mf.close()

    bounds_out = geom.bounds

    if is_ndvi:
        arr = merged[0].astype(float)
        arr[arr < -1.5] = np.nan
        arr[arr >  1.5] = np.nan
    else:
        arr = merged.transpose(1, 2, 0).astype(float)
        for c in range(arr.shape[2]):
            ch    = arr[..., c]
            valid = ch[ch > 0]
            if len(valid) > 0:
                p2, p98 = np.nanpercentile(valid, 2), np.nanpercentile(valid, 98)
                arr[..., c] = np.clip((ch - p2) / (p98 - p2 + 1e-10), 0, 1)

    return arr, bounds_out

print("✓ Funciones actualizadas — tile size fijo: 0.05°")

## 3 - Cargar la zona de estudio, en este caso Cisneros - Santo Domingo

In [ ]:
# Ruta al shapefile del municipio de Dabeiba
shp_path = "/home/ecastilloc/Descargas/WP.1.2.5/4_Febrero2026/data/municipios/CisnerosSquare.shp"

import os
if not os.path.exists(shp_path):
    raise FileNotFoundError(f"No se encontró el shapefile en: {shp_path}")

# Leer y reproyectar a WGS84
gdf = gpd.read_file(shp_path).to_crs(epsg=4326)
geom = gdf["geometry"].iloc[0]  # Tomar el primer (y único) polígono

# Convertir a ee.Geometry
aoi = shapely_to_ee(geom)

print(f"✓ Shapefile cargado: {shp_path}")
print(f"  CRS: {gdf.crs}")
print(f"  Tipo de geometría: {geom.geom_type}")
print(f"  Bounds: {geom.bounds}")

# Visualización rápida del AOI
fig, ax = plt.subplots(1, 1, figsize=(6, 5))
gdf.plot(ax=ax, edgecolor='red', facecolor='none', linewidth=2)
ax.set_title("Área de Interés - Seccion Àrea de estudio", fontsize=12)
ax.set_xlabel("Longitud")
ax.set_ylabel("Latitud")
plt.tight_layout()
plt.show()

## 4 - Definición de fechas y creación de mosaicos

In [ ]:
# Período 1: Pre-evento → ampliar de 3 a 6 meses
start_date1 = "2021-01-01"   # antes era 2020-06-13
end_date1   = "2021-04-30"

# Período 2: Post-evento → ampliar también
start_date2 = "2021-06-01"
end_date2   = "2021-09-30"   # antes era 2021-03-15

print("Fechas configuradas:")
print(f"  Mosaico PRE-evento:  {start_date1}  →  {end_date1}")
print(f"  Mosaico POST-evento: {start_date2}  →  {end_date2}")

print("\nCreando mosaico PRE-evento...")
img_pre = create_mosaic(start_date1, end_date1, aoi)

print("\nCreando mosaico POST-evento...")
img_post = create_mosaic(start_date2, end_date2, aoi)

# Diferencia NDVI: pre - post (valores positivos = pérdida de vegetación)
ndvi_diff = img_pre.select('NDVI').subtract(img_post.select('NDVI')).rename('NDVI_Diff')

print("\n✓ Mosaicos y diferencia NDVI calculados")

## 5 - Descarga de datos desde GEE para visualización

In [ ]:
SCALE = 10  # resolución nativa Sentinel-2

print("=" * 55)
print(f"Descargando a {SCALE}m/px por tiles — esto tomará muchos min")
print("=" * 55)

print("\n[1/5] RGB Pre-evento...")
rgb_pre_raw, bounds = download_mosaic_tiled(img_pre, geom, SCALE, ['B4','B3','B2'])

print("\n[2/5] RGB Post-evento...")
rgb_post_raw, _ = download_mosaic_tiled(img_post, geom, SCALE, ['B4','B3','B2'])

print("\n[3/5] NDVI Pre-evento...")
ndvi_pre_arr, _ = download_mosaic_tiled(img_pre, geom, SCALE, ['NDVI'], is_ndvi=True)

print("\n[4/5] NDVI Post-evento...")
ndvi_post_arr, _ = download_mosaic_tiled(img_post, geom, SCALE, ['NDVI'], is_ndvi=True)

print("\n[5/5] NDVI Diferencia...")
ndvi_diff_arr, _ = download_mosaic_tiled(ndvi_diff.rename('NDVI'), geom, SCALE, ['NDVI'], is_ndvi=True)

print("\n✓ Descarga completada")
print(f"  RGB shape:  {rgb_pre_raw.shape}")
print(f"  NDVI shape: {ndvi_pre_arr.shape}")

# Recalcular resolución efectiva
width_km  = (bounds[2] - bounds[0]) * 111 * np.cos(np.radians((bounds[1]+bounds[3])/2))
height_km = (bounds[3] - bounds[1]) * 111
h_arr, w_arr = ndvi_pre_arr.shape
print(f"  Resolución efectiva: {width_km*1000/w_arr:.0f} m/px × {height_km*1000/h_arr:.0f} m/px")

In [ ]:
width_km  = (bounds[2] - bounds[0]) * 111 * np.cos(np.radians((bounds[1]+bounds[3])/2))
height_km = (bounds[3] - bounds[1]) * 111
h_arr, w_arr = ndvi_diff_arr.shape
print(f"Resolución efectiva: {width_km*1000/w_arr:.1f} m/px × {height_km*1000/h_arr:.1f} m/px")

## 6 - Visualización RGB, NDVI y diff-NDVI

In [ ]:
# Normalizar RGB para visualización
def normalize_rgb_display(arr):
    """Normaliza cada canal RGB independientemente con percentiles 2-98"""
    if arr.shape[-1] == 4:  # RGBA → RGB
        arr = arr[..., :3]
    result = np.zeros_like(arr, dtype=float)
    for c in range(3):
        channel = arr[..., c].astype(float)
        p2, p98 = np.percentile(channel, 2), np.percentile(channel, 98)
        result[..., c] = np.clip((channel - p2) / (p98 - p2 + 1e-10), 0, 1)
    return result

rgb_pre_disp  = normalize_rgb_display(rgb_pre_raw)
rgb_post_disp = normalize_rgb_display(rgb_post_raw)

# Colormap para NDVI y diferencia
cmap_ndvi = plt.cm.RdYlGn
cmap_diff = plt.cm.RdYlGn_r  # Rojo = pérdida vegetación (positivo = pre > post)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Análisis NDVI Diferencial - Àrea de estudio\nDetección de posibles deslizamientos',
             fontsize=16, fontweight='bold', y=1.01)

# --- Panel 1: RGB Pre-evento ---
ax = axes[0, 0]
ax.imshow(rgb_pre_disp)
ax.set_title(f'RGB Pre-evento\n{start_date1} → {end_date1}', fontsize=11, fontweight='bold')
ax.axis('off')

# --- Panel 2: RGB Post-evento ---
ax = axes[0, 1]
ax.imshow(rgb_post_disp)
ax.set_title(f'RGB Post-evento\n{start_date2} → {end_date2}', fontsize=11, fontweight='bold')
ax.axis('off')

# --- Panel 3: Diferencia NDVI ---
ax = axes[0, 2]
vmax_diff = np.nanpercentile(np.abs(ndvi_diff_arr), 98)
im_diff = ax.imshow(ndvi_diff_arr, cmap=cmap_diff, vmin=-vmax_diff, vmax=vmax_diff)
ax.set_title('Diferencia NDVI\n(Pre − Post)  |  Rojo = pérdida vegetación', fontsize=11, fontweight='bold')
ax.axis('off')
plt.colorbar(im_diff, ax=ax, fraction=0.046, pad=0.04, label='ΔNDVI')

# --- Panel 4: NDVI Pre-evento ---
ax = axes[1, 0]
im_pre = ax.imshow(ndvi_pre_arr, cmap=cmap_ndvi, vmin=-0.2, vmax=0.9)
ax.set_title(f'NDVI Pre-evento\n{start_date1} → {end_date1}', fontsize=11, fontweight='bold')
ax.axis('off')
plt.colorbar(im_pre, ax=ax, fraction=0.046, pad=0.04, label='NDVI')

# --- Panel 5: NDVI Post-evento ---
ax = axes[1, 1]
im_post = ax.imshow(ndvi_post_arr, cmap=cmap_ndvi, vmin=-0.2, vmax=0.9)
ax.set_title(f'NDVI Post-evento\n{start_date2} → {end_date2}', fontsize=11, fontweight='bold')
ax.axis('off')
plt.colorbar(im_post, ax=ax, fraction=0.046, pad=0.04, label='NDVI')

# --- Panel 6: Histograma diferencia NDVI ---
ax = axes[1, 2]
valid_diff = ndvi_diff_arr[~np.isnan(ndvi_diff_arr)].ravel()
ax.hist(valid_diff, bins=100, color='steelblue', edgecolor='none', alpha=0.8)
_pcts_hist = [(PCT_NDVI, 'blue', '-')] + [
    t for t in [(90, 'orange', '--'), (95, 'red', '-.'), (99, 'darkred', ':')]
    if t[0] != PCT_NDVI]
for p, color, ls in _pcts_hist:
    val = np.percentile(valid_diff, p)
    lw  = 2.5 if p == PCT_NDVI else 2
    ax.axvline(val, color=color, linestyle=ls, linewidth=lw,
               label=f'P{p} = {val:.3f}' + (' (activo)' if p == PCT_NDVI else ''))
ax.axvline(0, color='black', linewidth=1, alpha=0.5)
ax.set_xlabel('ΔNDVI', fontsize=10)
ax.set_ylabel('Frecuencia', fontsize=10)
ax.set_title('Distribución de la Diferencia NDVI\n(con umbrales de percentil)', fontsize=11, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/dabeiba_ndvi_analisis.png', dpi=150, bbox_inches='tight')
plt.show()
print("✓ Figura guardada en /tmp/venecia_ndvi_analisis.png")

## 7 - Filtro por diferencia de NDVI usando umbrales de percentil (P90, P95, P99)

In [ ]:
valid_diff = ndvi_diff_arr[~np.isnan(ndvi_diff_arr)].ravel()
total_px   = np.sum(~np.isnan(ndvi_diff_arr))

# ─── Umbral ACTIVO (definido en la celda de configuración) ───
p_ndvi    = np.percentile(valid_diff, PCT_NDVI)
mask_ndvi = ndvi_diff_arr > p_ndvi

# Alias de compatibilidad: el resto del notebook usa p99 / mask_p99 / p95 / mask_p95.
# Todos apuntan ahora al MISMO umbral activo (antes P95 y P99 estaban mezclados).
p99  = p95  = p_ndvi
mask_p99 = mask_p95 = mask_ndvi

print(f"Umbral ACTIVO de pérdida de vegetación (ΔNDVI positivo = pre > post):")
print(f"  Percentil {PCT_NDVI}: ΔNDVI > {p_ndvi:.4f}")
print(f"  Píxeles candidatos: {np.sum(mask_ndvi):,} ({100*np.sum(mask_ndvi)/total_px:.1f}% del área)")

# ─── Percentiles de referencia (comparación y validación) ────
PCTS_REF     = sorted({PCT_NDVI, 90, 95, 99})
umbrales_ref = {p: float(np.percentile(valid_diff, p)) for p in PCTS_REF}
masks_ref    = {p: ndvi_diff_arr > umbrales_ref[p] for p in PCTS_REF}

# Alias de compatibilidad para celdas que esperan mask_p90
mask_p90 = masks_ref[90]
p90      = umbrales_ref[90]

PCTS_COMPARA = sorted({PCT_NDVI, 90, 99})
print(f"\nPercentiles de referencia: {PCTS_REF}")
for p in PCTS_REF:
    m = masks_ref[p]
    marca = '  ← ACTIVO' if p == PCT_NDVI else ''
    print(f"  P{p}: ΔNDVI > {umbrales_ref[p]:.4f}  →  {np.sum(m):,} px "
          f"({100*np.sum(m)/total_px:.1f}% del área){marca}")

# ─── Visualización comparativa ───────────────────────────────
n_pan = len(PCTS_COMPARA)
fig, axes = plt.subplots(1, n_pan, figsize=(6*n_pan, 6))
axes = np.atleast_1d(axes)
fig.suptitle(
    f'Zonas de Posible Deslizamiento - Umbral por Percentil de ΔNDVI\n'
    f'Àrea de estudio  |  Umbral activo: P{PCT_NDVI}  (Rojo = pérdida significativa de vegetación)',
    fontsize=14, fontweight='bold'
)

colores = {p: c for p, c in zip(PCTS_COMPARA, ['orangered', 'red', 'darkred', 'maroon'])}

for ax, pct in zip(axes, PCTS_COMPARA):
    threshold = umbrales_ref[pct]
    mask      = masks_ref[pct]
    ax.imshow(ndvi_diff_arr, cmap='Greys',
              vmin=np.nanmin(ndvi_diff_arr),
              vmax=np.nanpercentile(ndvi_diff_arr, 99), alpha=0.6)
    overlay = np.zeros((*ndvi_diff_arr.shape, 4), dtype=float)
    overlay[mask] = [0.9, 0.1, 0.1, 0.85]
    ax.imshow(overlay)
    px_count = np.sum(mask)
    pct_area = 100 * px_count / total_px
    activo   = '  ★ ACTIVO' if pct == PCT_NDVI else ''
    ax.set_title(f'Umbral Percentil {pct}{activo}\nΔNDVI > {threshold:.3f}\n'
                 f'{px_count:,} píxeles ({pct_area:.1f}% del área)',
                 fontsize=11, fontweight='bold')
    ax.axis('off')

plt.tight_layout()


In [ ]:
# ─── Visualización comparativa (sin título de figura) ────────
n_pan = len(PCTS_COMPARA)
h_img, w_img = ndvi_diff_arr.shape
PANEL_W = 6.5
PANEL_H = PANEL_W / (w_img / h_img)

fig, axes = plt.subplots(1, n_pan, figsize=(PANEL_W * n_pan, PANEL_H))
axes = np.atleast_1d(axes)

for ax, pct in zip(axes, PCTS_COMPARA):
    threshold = umbrales_ref[pct]
    mask      = masks_ref[pct]
    ax.imshow(ndvi_diff_arr, cmap='Greys',
              vmin=np.nanmin(ndvi_diff_arr),
              vmax=np.nanpercentile(ndvi_diff_arr, 99), alpha=0.6)
    overlay = np.zeros((*ndvi_diff_arr.shape, 4), dtype=float)
    overlay[mask] = [0.9, 0.1, 0.1, 0.85]
    ax.imshow(overlay)
    px_count = np.sum(mask)
    pct_area = 100 * px_count / total_px
    activo   = '  ★ ACTIVO' if pct == PCT_NDVI else ''
    ax.set_title(f'Umbral Percentil {pct}{activo}\nΔNDVI > {threshold:.3f}\n'
                 f'{px_count:,} píxeles ({pct_area:.1f}% del área)',
                 fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()

# ─── Guardar ─────────────────────────────────────────────────
OUT_DIR = '/home/ecastilloc/Descargas/WP.1.2.5/figuras'
os.makedirs(OUT_DIR, exist_ok=True)
out_png = os.path.join(OUT_DIR, 'Figure6.png')
plt.savefig(out_png, dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(out_png.replace('.png', '.pdf'), bbox_inches='tight', facecolor='white')
plt.show()

print(f"\n✓ Guardado en {out_png}  (+ versión .pdf)")

## 8 - Filtro BSI (Bare Soil Index)

In [ ]:
# ============================================================
# 8b - Filtro BSI (Bare Soil Index)
# BSI = ((B11+B4)-(B8+B2)) / ((B11+B4)+(B8+B2))
# Umbral ACTIVO: percentil PCT_BSI del ΔBSI calculado SOBRE los
# candidatos NDVI (mask_ndvi). Los demás son solo referencia.
# ============================================================

print("Calculando y descargando BSI desde GEE...")

# ─── 1. Calcular BSI en GEE ─────────────────────────────────
def compute_bsi(image):
    return image.addBands(
        image.expression(
            '((SWIR + RED) - (NIR + BLUE)) / ((SWIR + RED) + (NIR + BLUE) + 1e-10)',
            {'SWIR': image.select('B11'), 'RED': image.select('B4'),
             'NIR':  image.select('B8'),  'BLUE': image.select('B2')}
        ).rename('BSI')
    )

img_pre_bsi  = compute_bsi(img_pre)
img_post_bsi = compute_bsi(img_post)
bsi_diff_gee = img_post_bsi.select('BSI').subtract(
               img_pre_bsi.select('BSI')).rename('NDVI')

# ─── 2. Descargar por tiles ──────────────────────────────────
print("  ↓ BSI diferencia...")
bsi_diff_arr, _ = download_mosaic_tiled(bsi_diff_gee, geom, SCALE, ['NDVI'], is_ndvi=True)

print(f"  BSI diff — shape: {bsi_diff_arr.shape}")
print(f"  Rango: [{np.nanmin(bsi_diff_arr):.3f}, {np.nanmax(bsi_diff_arr):.3f}]")

# ─── 3. Umbrales BSI sobre los candidatos NDVI ───────────────
bsi_en_cand = bsi_diff_arr[mask_ndvi & ~np.isnan(bsi_diff_arr)]

BSI_ACT = float(np.percentile(bsi_en_cand, PCT_BSI))   # ← umbral ACTIVO
BSI_P25 = BSI_ACT                                      # alias de compatibilidad

# Percentiles de referencia para la comparación visual
PCTS_BSI_COMPARA = sorted({25, 50, PCT_BSI})
BSI_UMBRALES = {p: float(np.percentile(bsi_en_cand, p)) for p in PCTS_BSI_COMPARA}
BSI_UMBRALES[0] = 0.0   # escenario "cualquier exposición de suelo"

print(f"\nUmbrales BSI calculados sobre los {np.sum(mask_ndvi):,} candidatos NDVI (P{PCT_NDVI}):")
print(f"  Escenario BSI > 0: 0.0000  — más permisivo")
for p in PCTS_BSI_COMPARA:
    marca = '  ← ACTIVO' if p == PCT_BSI else ''
    print(f"  Escenario BSI > P{p}: {BSI_UMBRALES[p]:.4f}{marca}")

# ─── 4. Máscaras por escenario ──────────────────────────────
escenarios = [(0, 'BSI > 0\n(permisivo)', 0.0)] + \
             [(p, f'BSI > P{p}', BSI_UMBRALES[p]) for p in PCTS_BSI_COMPARA]

masks_bsi = {p: mask_ndvi & (bsi_diff_arr > u) for p, _, u in escenarios}

n_cand = max(np.sum(mask_ndvi), 1)
print(f"\nPíxeles retenidos por escenario (base NDVI P{PCT_NDVI} = {np.sum(mask_ndvi):,} px):")
for p, _, u in escenarios:
    etiqueta = 'BSI > 0  ' if p == 0 else f'BSI > P{p}'
    print(f"  {etiqueta}: {np.sum(masks_bsi[p]):,} px  "
          f"({100*np.sum(masks_bsi[p])/n_cand:.1f}%)")

# ─── 5. Visualización comparativa ───────────────────────────
n_pan = len(escenarios) + 1
fig, axes = plt.subplots(1, n_pan, figsize=(5.5*n_pan, 6))
fig.suptitle(f'Filtro BSI — Escenarios de umbral (activo: P{PCT_BSI})\n'
             'Àrea de estudio', fontsize=13, fontweight='bold')

# Panel 1: ΔBSI completo
ax = axes[0]
vmax_bsi = np.nanpercentile(np.abs(bsi_diff_arr), 98)
im = ax.imshow(bsi_diff_arr, cmap='YlOrBr', vmin=-vmax_bsi, vmax=vmax_bsi)
ax.set_title('ΔBSI (Post − Pre)\n(marrón = más suelo desnudo)',
             fontsize=10, fontweight='bold')
ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='ΔBSI')

from matplotlib.patches import Patch
for ax, (p, titulo, umbral) in zip(axes[1:], escenarios):
    mask_esc = masks_bsi[p]

    ax.imshow(ndvi_diff_arr, cmap='Greys', alpha=0.4,
              vmin=np.nanpercentile(ndvi_diff_arr, 2),
              vmax=np.nanpercentile(ndvi_diff_arr, 98))

    ov_desc = np.zeros((*mask_ndvi.shape, 4), dtype=float)
    ov_desc[mask_ndvi & ~mask_esc] = [0.5, 0.5, 0.5, 0.5]
    ax.imshow(ov_desc)

    ov_ret = np.zeros((*mask_esc.shape, 4), dtype=float)
    ov_ret[mask_esc] = [0.85, 0.05, 0.05, 0.9]
    ax.imshow(ov_ret)

    pct    = 100 * np.sum(mask_esc) / n_cand
    activo = '  ★ ACTIVO' if p == PCT_BSI else ''
    ax.set_title(f'{titulo}{activo}\nUmbral: {umbral:.3f}\n'
                 f'{np.sum(mask_esc):,} px ({pct:.1f}%)',
                 fontsize=10, fontweight='bold')
    ax.axis('off')
    ax.legend(handles=[
        Patch(facecolor='red',  alpha=0.9, label=f'Retenidos: {np.sum(mask_esc):,}'),
        Patch(facecolor='gray', alpha=0.5, label=f'Descartados: {np.sum(mask_ndvi&~mask_esc):,}'),
    ], loc='lower right', fontsize=8, framealpha=0.9)

plt.tight_layout()
plt.savefig('/tmp/filtro_bsi_escenarios.png', dpi=150, bbox_inches='tight')
plt.show()

# ─── 6. Escenario activo ────────────────────────────────────
mask_bsi     = masks_bsi[PCT_BSI]
mask_p99_bsi = mask_bsi          # alias usado aguas abajo

print(f"\n✓ Escenario activo: ΔBSI > P{PCT_BSI} ({BSI_ACT:.4f}) → {np.sum(mask_p99_bsi):,} píxeles")
print(f"  Para cambiarlo, modifica PCT_BSI en la celda de configuración")
print("✓ Guardado en /tmp/filtro_bsi_escenarios.png")


## 9 - Filtro de pendientes (> 15°) 
### Resaltando solo las zonas de ladera con pérdida de vegetación

In [ ]:
from scipy.ndimage import zoom as ndimage_zoom

# Ruta al raster de pendientes
slope_path = "/home/ecastilloc/Descargas/WP.1.2.5/4_Febrero2026/data/dem/Pendiente_CisnerosSquare.tif"

if not os.path.exists(slope_path):
    raise FileNotFoundError(f"No se encontró el raster de pendientes en: {slope_path}")

# Leer raster de pendientes
with rasterio.open(slope_path) as src:
    slope_arr   = src.read(1).astype(float)
    slope_nodata = src.nodata
    slope_bounds = src.bounds
    slope_crs    = src.crs
    slope_transform = src.transform
    print(f"✓ Raster de pendientes cargado")
    print(f"  Resolución: {slope_arr.shape}")
    print(f"  CRS: {slope_crs}")
    print(f"  Bounds: {slope_bounds}")
    print(f"  Rango de valores: {np.nanmin(slope_arr):.1f}° - {np.nanmax(slope_arr):.1f}°")

# Aplicar nodata
if slope_nodata is not None:
    slope_arr[slope_arr == slope_nodata] = np.nan

# === Remuestrear pendiente al mismo tamaño que NDVI diff ===
# (para poder aplicar máscara directamente)
h_ndvi, w_ndvi = ndvi_diff_arr.shape
h_slope, w_slope = slope_arr.shape

zoom_y = h_ndvi / h_slope
zoom_x = w_ndvi / w_slope

# Remuestrear con scipy zoom (interpolación bilineal)
slope_resampled = ndimage_zoom(np.nan_to_num(slope_arr, nan=0), (zoom_y, zoom_x), order=1)

# Donde había nan, poner nan de nuevo
nan_mask_orig = np.isnan(slope_arr)
nan_mask_resamp = ndimage_zoom(nan_mask_orig.astype(float), (zoom_y, zoom_x), order=0) > 0.5
slope_resampled[nan_mask_resamp] = np.nan

print(f"\n✓ Pendiente remuestreada a {slope_resampled.shape} para coincidir con NDVI diff")

# === Máscara de ladera: pendiente > 15° ===
SLOPE_THRESHOLD = 15.0  # grados
ladera_mask = slope_resampled > SLOPE_THRESHOLD
print(f"  Píxeles con pendiente > {SLOPE_THRESHOLD}°: {np.sum(ladera_mask):,} "
      f"({100*np.sum(ladera_mask)/ladera_mask.size:.1f}% del área)")

In [ ]:
# ============================================================
# 9 - Máscara de pendientes + Diferencia NDVI
# CONFIGURABLE: cambia solo estas variables para otra área
# ============================================================

SLOPE_THRESHOLD = 15  # grados — umbral de pendiente mínima

# Nombre del municipio desde el shapefile (busca columnas comunes)
col_nombre = next((c for c in gdf.columns
                   if c.upper() in ['NOMBRE','NAME','MUNICIPIO','MPIO','NOM_MPIO']), None)
NOMBRE_AREA = gdf[col_nombre].iloc[0] if col_nombre else 'Área de estudio'

print(f"Área de estudio: {NOMBRE_AREA}")
print(f"Umbral de pendiente: {SLOPE_THRESHOLD}°")

# ─── Aplicar máscara de ladera ───────────────────────────────
ndvi_diff_ladera = ndvi_diff_arr.copy()
ndvi_diff_ladera[~ladera_mask] = np.nan

# ─── Figura 4 paneles ────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 13))
fig.suptitle(
    f'Máscara de Pendientes (> {SLOPE_THRESHOLD}°) × Diferencia NDVI\n'
    f'{NOMBRE_AREA} — Identificación de posibles deslizamientos en ladera',
    fontsize=14, fontweight='bold'
)

# --- Panel 1: Mapa de pendientes ---
ax = axes[0, 0]
im_slope = ax.imshow(slope_resampled, cmap='terrain',
                     vmin=0, vmax=np.nanpercentile(slope_resampled, 99))
ax.contour(ladera_mask.astype(int), levels=[0.5],
           colors='red', linewidths=0.8, alpha=0.7)
ax.set_title(f'Mapa de Pendientes (grados)\n(Contorno rojo: > {SLOPE_THRESHOLD}°)',
             fontsize=11, fontweight='bold')
ax.axis('off')
plt.colorbar(im_slope, ax=ax, fraction=0.046, pad=0.04, label='Pendiente (°)')

# --- Panel 2: NDVI diff completa ---
ax = axes[0, 1]
vmax_diff = np.nanpercentile(np.abs(ndvi_diff_arr), 98)
im_full   = ax.imshow(ndvi_diff_arr, cmap='RdYlGn_r', vmin=-vmax_diff, vmax=vmax_diff)
ax.set_title('Diferencia NDVI completa\n(Pre − Post | sin filtro de pendiente)',
             fontsize=11, fontweight='bold')
ax.axis('off')
plt.colorbar(im_full, ax=ax, fraction=0.046, pad=0.04, label='ΔNDVI')

# --- Panel 3: NDVI diff en laderas ---
ax = axes[1, 0]
flat_vis = np.where(ladera_mask, np.nan, ndvi_diff_arr)
ax.imshow(flat_vis, cmap='Greys',
          vmin=np.nanmin(ndvi_diff_arr), vmax=np.nanmax(ndvi_diff_arr), alpha=0.25)
im_ladera = ax.imshow(ndvi_diff_ladera, cmap='RdYlGn_r', vmin=-vmax_diff, vmax=vmax_diff)
ax.set_title(f'Diferencia NDVI en laderas (> {SLOPE_THRESHOLD}°)\n'
             f'(Zonas planas en gris/transparente)',
             fontsize=11, fontweight='bold')
ax.axis('off')
plt.colorbar(im_ladera, ax=ax, fraction=0.046, pad=0.04, label='ΔNDVI en ladera')

# --- Panel 4: Umbrales P90/P95/P99 en laderas ---
ax = axes[1, 1]
valid_ladera = ndvi_diff_ladera[~np.isnan(ndvi_diff_ladera)].ravel()
p90_l = np.percentile(valid_ladera, 90)
p95_l = np.percentile(valid_ladera, 95)
p99_l = np.percentile(valid_ladera, 99)

ax.imshow(slope_resampled, cmap='Greys',
          vmin=0, vmax=np.nanpercentile(slope_resampled, 99), alpha=0.5)

for threshold, rgba, label in [
    (p90_l, (1.0, 0.6, 0.0, 0.5),  'P90'),
    (p95_l, (0.9, 0.1, 0.1, 0.65), 'P95'),
    (p99_l, (0.5, 0.0, 0.0, 0.85), 'P99'),
]:
    m       = ndvi_diff_ladera > threshold
    overlay = np.zeros((*ndvi_diff_ladera.shape, 4), dtype=float)
    overlay[m] = rgba
    ax.imshow(overlay)

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='orange',  alpha=0.7, label=f'P90: ΔNDVI > {p90_l:.3f}'),
    Patch(facecolor='red',     alpha=0.8, label=f'P95: ΔNDVI > {p95_l:.3f}'),
    Patch(facecolor='darkred', alpha=0.9, label=f'P99: ΔNDVI > {p99_l:.3f}'),
], loc='lower right', fontsize=9, framealpha=0.85, title='Umbral (en ladera)')
ax.set_title(
    f'Posibles deslizamientos en ladera\n'
    f'(P90/P95/P99 del ΔNDVI en pendiente > {SLOPE_THRESHOLD}°)',
    fontsize=11, fontweight='bold'
)
ax.axis('off')

plt.tight_layout()
nombre_archivo = NOMBRE_AREA.lower().replace(' ', '_')
plt.savefig(f'/tmp/{nombre_archivo}_laderas_deslizamientos.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Guardado en /tmp/{nombre_archivo}_laderas_deslizamientos.png")
print(f"\nUmbrales ΔNDVI en laderas (pendiente > {SLOPE_THRESHOLD}°):")
print(f"  P90: {p90_l:.4f}  →  {np.sum(ndvi_diff_ladera > p90_l):,} píxeles")
print(f"  P95: {p95_l:.4f}  →  {np.sum(ndvi_diff_ladera > p95_l):,} píxeles")
print(f"  P99: {p99_l:.4f}  →  {np.sum(ndvi_diff_ladera > p99_l):,} píxeles")

## 10 - Filtro Curvatura del terreno
### Zonas cóncavas acumulan material; se descartan zonas convexas

In [ ]:
# ============================================================
# 10 - Filtro Curvatura del terreno
# CONFIGURABLE: la ruta del DEM se construye automáticamente
# ============================================================

from scipy.ndimage import laplace, zoom as ndimage_zoom
import os

print(f"Área de estudio: {NOMBRE_AREA}")
print("Calculando curvatura del terreno...")

# ─── 1. Cargar DEM — rutas automáticas desde slope_path ──────
dem_candidatos = [
    # Variaciones automáticas basadas en slope_path
    slope_path.replace('Pendiente_', 'DEM_'),
    slope_path.replace('Pendiente_', 'MDT_'),
    slope_path.replace('Pendiente_', 'dem_'),
    slope_path.replace('Pendiente', 'DEM'),
    slope_path.replace('Pendiente', 'dem'),
    slope_path.replace('Pendiente', 'MDT'),
    # Carpeta dem con nombre del área
    os.path.join(os.path.dirname(slope_path), f'DEM_{NOMBRE_AREA}.tif'),
    os.path.join(os.path.dirname(slope_path), f'dem_{NOMBRE_AREA}.tif'),
    os.path.join(os.path.dirname(slope_path), f'{NOMBRE_AREA}.tif'),
]

dem_arr_raw = None
for dp in dem_candidatos:
    if os.path.exists(dp):
        print(f"  ✓ DEM encontrado: {dp}")
        with rasterio.open(dp) as src:
            dem_arr_raw = src.read(1).astype(float)
            if src.nodata is not None:
                dem_arr_raw[dem_arr_raw == src.nodata] = np.nan
        break

if dem_arr_raw is None:
    print("  ⚠ DEM no encontrado automáticamente")
    print("  Rutas buscadas:")
    for dp in dem_candidatos:
        print(f"    {dp}")
    print("  → Usando mapa de pendientes como aproximación")
    dem_arr_raw = slope_resampled.copy()
else:
    # Remuestrear DEM al grid NDVI
    h_ndvi, w_ndvi = ndvi_diff_arr.shape
    h_dem,  w_dem  = dem_arr_raw.shape
    dem_arr_raw = ndimage_zoom(
        np.nan_to_num(dem_arr_raw, nan=0),
        (h_ndvi / h_dem, w_ndvi / w_dem), order=1
    )

# ─── 2. Calcular curvatura con Laplaciano ────────────────────
curvatura_arr = laplace(np.nan_to_num(dem_arr_raw, nan=0))

print(f"  Rango curvatura: [{np.nanmin(curvatura_arr):.4f}, {np.nanmax(curvatura_arr):.4f}]")
print(f"  Píxeles cóncavos (< 0): {np.sum(curvatura_arr < 0):,}")
print(f"  Píxeles convexos (> 0): {np.sum(curvatura_arr > 0):,}")

# ─── 3. Aplicar filtro sobre candidatos P_NDVI+BSI+Pendiente ────
mask_activos    = mask_p99_bsi & ladera_mask
curv_en_activos = curvatura_arr[mask_activos]
CURV_UMBRAL     = np.percentile(curv_en_activos, 85)
mask_curvatura  = curvatura_arr <= CURV_UMBRAL

mask_p99_bsi_slope_curv = mask_p99_bsi & ladera_mask & mask_curvatura

print(f"\nFiltro Curvatura aplicado:")
print(f"  Umbral curvatura <= {CURV_UMBRAL:.4f} (excluye 15% más convexo)")
print(f"  Antes  ({LBL_NDVI}+BSI+Pendiente): {np.sum(mask_activos):,} px")
print(f"  Después (+Curvatura):       {np.sum(mask_p99_bsi_slope_curv):,} px "
      f"({100*np.sum(mask_p99_bsi_slope_curv)/max(np.sum(mask_activos),1):.1f}% retenidos)")

# ─── 4. Visualización ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(19, 6))
fig.suptitle(f'Filtro Curvatura — Zonas cóncavas vs convexas\n{NOMBRE_AREA}',
             fontsize=13, fontweight='bold')

# Panel 1: mapa de curvatura
ax = axes[0]
vmax_c = np.nanpercentile(np.abs(curvatura_arr), 98)
im = ax.imshow(curvatura_arr, cmap='RdBu_r', vmin=-vmax_c, vmax=vmax_c)
ax.set_title('Curvatura del terreno\n(azul=cóncavo  |  rojo=convexo)',
             fontsize=10, fontweight='bold')
ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Curvatura')

# Panel 2: antes del filtro
ax = axes[1]
ax.imshow(slope_resampled, cmap='terrain',
          vmin=0, vmax=np.nanpercentile(slope_resampled, 99), alpha=0.5)
ov = np.zeros((*mask_activos.shape, 4), dtype=float)
ov[mask_activos] = [0.85, 0.05, 0.05, 0.85]
ax.imshow(ov)
ax.set_title(f'Antes ({LBL_NDVI}+BSI+Pendiente)\n{np.sum(mask_activos):,} píxeles',
             fontsize=10, fontweight='bold')
ax.axis('off')

# Panel 3: después del filtro
ax = axes[2]
ax.imshow(slope_resampled, cmap='terrain',
          vmin=0, vmax=np.nanpercentile(slope_resampled, 99), alpha=0.5)
ov_desc = np.zeros((*mask_activos.shape, 4), dtype=float)
ov_desc[mask_activos & ~mask_curvatura] = [0.5, 0.5, 0.5, 0.5]
ax.imshow(ov_desc)
ov_ret = np.zeros((*mask_p99_bsi_slope_curv.shape, 4), dtype=float)
ov_ret[mask_p99_bsi_slope_curv] = [0.85, 0.05, 0.05, 0.9]
ax.imshow(ov_ret)
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor='red',  alpha=0.9, label=f'Retenidos: {np.sum(mask_p99_bsi_slope_curv):,} px'),
    Patch(facecolor='gray', alpha=0.6, label=f'Descartados: {np.sum(mask_activos & ~mask_curvatura):,} px'),
], loc='lower right', fontsize=9, framealpha=0.9)
ax.set_title(f'Después (+Curvatura)\n{np.sum(mask_p99_bsi_slope_curv):,} píxeles retenidos',
             fontsize=10, fontweight='bold')
ax.axis('off')

plt.tight_layout()
nombre_archivo = NOMBRE_AREA.lower().replace(' ', '_')
plt.savefig(f'/tmp/{nombre_archivo}_filtro_curvatura.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"✓ Guardado en /tmp/{nombre_archivo}_filtro_curvatura.png")

## 11 - Ver los umbrales

In [ ]:
# === Mapa resumen — efecto acumulativo de cada filtro ===

from matplotlib.patches import Patch

fig, axes = plt.subplots(2, 3, figsize=(22, 14))
fig.suptitle(
    'Pipeline de filtros — Reducción acumulativa de candidatos\n'
    'Àrea de estudio',
    fontsize=16, fontweight='bold'
)

# Aplanar axes para iterar igual que antes
axes_flat = axes.ravel()

# Definir etapas acumulativas (sin susceptibilidad)
etapas = [
    (mask_p99,
     f'Filtro 1 — NDVI {LBL_NDVI}',
     f'ΔNDVI > {p99:.3f}\n{np.sum(mask_p99):,} px  (100%)'),  # ← coma aquí
    (mask_p99_bsi,
     'Filtro 2 — + BSI',
     f'ΔBSI > {BSI_ACT:.3f} ({LBL_BSI})\n{np.sum(mask_p99_bsi):,} px  '
     f'(−{np.sum(mask_p99)-np.sum(mask_p99_bsi):,} removidos)'),
    (mask_p99_bsi & ladera_mask,
     'Filtro 3 — + Pendiente',
     f'Pendiente > {SLOPE_THRESHOLD}°\n{np.sum(mask_p99_bsi & ladera_mask):,} px  '
     f'(−{np.sum(mask_p99_bsi)-np.sum(mask_p99_bsi & ladera_mask):,} removidos)'),
    (mask_p99_bsi_slope_curv,
     'Filtro 4 — + Curvatura',
     f'Excluye 15% más convexo\n{np.sum(mask_p99_bsi_slope_curv):,} px  '
     f'(−{np.sum(mask_p99_bsi & ladera_mask)-np.sum(mask_p99_bsi_slope_curv):,} removidos)'),
]

fondos      = [ndvi_diff_arr, bsi_diff_arr, slope_resampled, curvatura_arr]
fondo_cmaps = ['RdYlGn_r',   'YlOrBr',     'terrain',       'RdBu_r'     ]
mascaras    = [m for m, *_ in etapas]

for i, (ax, (mask_e, titulo, subtitulo), fondo, fcmap) in \
        enumerate(zip(axes_flat, etapas, fondos, fondo_cmaps)):

    vmin_f = 0 if fcmap == 'terrain' else np.nanpercentile(fondo, 2)
    vmax_f = np.nanpercentile(fondo, 98)
    ax.imshow(fondo, cmap=fcmap, vmin=vmin_f, vmax=vmax_f, alpha=0.55)

    if i > 0:
        mask_prev = mascaras[i - 1]
        ov_desc = np.zeros((*mask_e.shape, 4), dtype=float)
        ov_desc[mask_prev & ~mask_e] = [0.4, 0.4, 0.4, 0.55]
        ax.imshow(ov_desc)

    ov_ret = np.zeros((*mask_e.shape, 4), dtype=float)
    ov_ret[mask_e] = [0.85, 0.05, 0.05, 0.92]
    ax.imshow(ov_ret)

    pct_orig  = 100 * np.sum(mask_e) / np.sum(mask_p95)
    removidos = np.sum(mascaras[i-1] & ~mask_e) if i > 0 else 0

    ax.set_title(f'{titulo}\n{subtitulo}\n({pct_orig:.1f}% del {LBL_NDVI} original)',
                 fontsize=11, fontweight='bold')
    ax.axis('off')
    ax.legend(handles=[
        Patch(facecolor='red',  alpha=0.9, label=f'Retenidos: {np.sum(mask_e):,}'),
        Patch(facecolor='gray', alpha=0.6, label=f'Removidos: {removidos:,}'),
    ], loc='lower right', fontsize=9, framealpha=0.9)

# Panel 5: gráfico de barras resumen
ax = axes_flat[4]
ax.set_facecolor('#f8f8f8')
nombres = [f'NDVI\n{LBL_NDVI}', f'+BSI\n{LBL_BSI}', '+Pendiente\n>15°', '+Curvatura\nP85']
valores = [np.sum(m) for m in mascaras]
colores = ['#d73027', '#fc8d59', '#4dac26', '#4575b4']

bars = ax.bar(nombres, valores, color=colores, edgecolor='white', linewidth=1.5, width=0.6)

for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{val:,}', ha='center', va='bottom', fontsize=10, fontweight='bold')

for i in range(len(valores) - 1):
    removidos_i = valores[i] - valores[i+1]
    pct_i = 100 * removidos_i / valores[i]
    ax.annotate(
        f'−{removidos_i:,}\n({pct_i:.0f}%)',
        xy=(i + 0.7, valores[i+1] + (valores[i] - valores[i+1]) / 2),
        fontsize=9, color='gray', ha='left', va='center'
    )

ax.set_title('Reducción acumulativa\npor filtro aplicado',
             fontsize=11, fontweight='bold')
ax.set_ylabel('Número de píxeles', fontsize=11)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.grid(True, axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

# Panel 6: ocultar (vacío)
axes_flat[5].axis('off')

plt.tight_layout(pad=2.0)
plt.savefig('/tmp/venecia_pipeline_filtros.png', dpi=180, bbox_inches='tight')
plt.show()

print("✓ Guardado en /tmp/venecia_pipeline_filtros.png")
print(f"\nResumen pipeline:")
for (mask_e, titulo, _), val in zip(etapas, valores):
    print(f"  {titulo:35s}: {val:,} px")
reduccion = np.sum(mask_p95) - np.sum(mask_p99_bsi_slope_curv)
print(f"  {'Reducción total (4 filtros)':35s}: {reduccion:,} px "
      f"({100*reduccion/np.sum(mask_p99):.1f}%)")

In [ ]:
from pyproj import Transformer
from matplotlib.patches import Patch

# ============================================================
# CONFIGURABLE: coordenadas de zoom en EPSG:3857
# Opción A: dejar en None para zoom automático al centroide
# Opción B: poner coordenadas específicas de la zona de interés
# ============================================================
ZOOM_X_MIN = None  # ejemplo: -8429149
ZOOM_X_MAX = None  # ejemplo: -8427177
ZOOM_Y_MIN = None  # ejemplo: 662003
ZOOM_Y_MAX = None  # ejemplo: 663520
ZOOM_DELTA_M = 2000  # buffer en metros si zoom automático

# === Calcular coordenadas de zoom ============================
transformer_fwd = Transformer.from_crs("EPSG:3857", "EPSG:4326", always_xy=True)
transformer_inv = Transformer.from_crs("EPSG:4326", "EPSG:3857", always_xy=True)

if ZOOM_X_MIN is None:
    # Zoom automático al centroide del AOI
    cx = (bounds[0] + bounds[2]) / 2
    cy = (bounds[1] + bounds[3]) / 2
    cx_3857, cy_3857 = transformer_inv.transform(cx, cy)
    x_min_z = cx_3857 - ZOOM_DELTA_M
    x_max_z = cx_3857 + ZOOM_DELTA_M
    y_min_z = cy_3857 - ZOOM_DELTA_M
    y_max_z = cy_3857 + ZOOM_DELTA_M
    print(f"Zoom automático al centroide de {NOMBRE_AREA}")
else:
    x_min_z, x_max_z = ZOOM_X_MIN, ZOOM_X_MAX
    y_min_z, y_max_z = ZOOM_Y_MIN, ZOOM_Y_MAX
    print(f"Zoom manual a coordenadas especificadas")

# Convertir a WGS84
lon_min_z, lat_min_z = transformer_fwd.transform(x_min_z, y_min_z)
lon_max_z, lat_max_z = transformer_fwd.transform(x_max_z, y_max_z)
print(f"  Lon: {lon_min_z:.5f} → {lon_max_z:.5f}")
print(f"  Lat: {lat_min_z:.5f} → {lat_max_z:.5f}")

# === Convertir bbox a índices de pixel =======================
lon_min_aoi, lat_min_aoi, lon_max_aoi, lat_max_aoi = bounds
h, w = ndvi_diff_arr.shape

def geo_to_pixel(lon, lat):
    px = int((lon - lon_min_aoi) / (lon_max_aoi - lon_min_aoi) * w)
    py = int((lat_max_aoi - lat) / (lat_max_aoi - lat_min_aoi) * h)
    return px, py

px_min, py_min = geo_to_pixel(lon_min_z, lat_max_z)
px_max, py_max = geo_to_pixel(lon_max_z, lat_min_z)
px_min = max(0, px_min); py_min = max(0, py_min)
px_max = min(w, px_max); py_max = min(h, py_max)

print(f"  Píxeles: x[{px_min}:{px_max}]  y[{py_min}:{py_max}]")

# Verificar tamaño
if px_max <= px_min or py_max <= py_min:
    raise ValueError(
        f"Zoom fuera del AOI. Verifica coordenadas o aumenta ZOOM_DELTA_M.\n"
        f"AOI bounds: {bounds}"
    )

# === Recortar arrays =========================================
ndvi_diff_zoom    = ndvi_diff_arr[py_min:py_max, px_min:px_max]
mask_filtros_zoom = mask_p99_bsi_slope_curv[py_min:py_max, px_min:px_max]
slope_zoom        = slope_resampled[py_min:py_max, px_min:px_max]

def crop_rgb(rgb_arr):
    h_rgb, w_rgb = rgb_arr.shape[:2]
    return rgb_arr[
        int(py_min * h_rgb / h):int(py_max * h_rgb / h),
        int(px_min * w_rgb / w):int(px_max * w_rgb / w),
        :3
    ]

rgb_pre_zoom  = crop_rgb(rgb_pre_disp)
rgb_post_zoom = crop_rgb(rgb_post_disp)

print(f"  Shape zoom: ndvi={ndvi_diff_zoom.shape}  rgb={rgb_pre_zoom.shape}")

# === Plot 2×2 ================================================
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle(
    f'Acercamiento — {NOMBRE_AREA}\n'
    f'Sentinel-2 pre/post  +  ΔNDVI crudo vs candidatos tras pipeline de filtros',
    fontsize=14, fontweight='bold'
)

# Panel 1: RGB Pre
ax = axes[0, 0]
ax.imshow(rgb_pre_zoom)
ax.set_title(f'RGB Pre-evento\n{start_date1} → {end_date1}',
             fontsize=12, fontweight='bold')
ax.axis('off')

# Panel 2: RGB Post
ax = axes[0, 1]
ax.imshow(rgb_post_zoom)
ax.set_title(f'RGB Post-evento\n{start_date2} → {end_date2}',
             fontsize=12, fontweight='bold')
ax.axis('off')

# Panel 3: ΔNDVI crudo + P_NDVI
ax = axes[1, 0]
vmax_z = np.nanpercentile(np.abs(ndvi_diff_zoom), 98)
im     = ax.imshow(ndvi_diff_zoom, cmap='RdYlGn_r', vmin=-vmax_z, vmax=vmax_z)
cbar   = plt.colorbar(im, ax=ax, fraction=0.035, pad=0.03, shrink=0.7)
cbar.set_label('ΔNDVI', fontsize=11)
norm_p99 = (p99 + vmax_z) / (2 * vmax_z)
cbar.ax.axhline(norm_p99, color='darkred', linewidth=2, linestyle='--')
cbar.ax.text(1.1, norm_p99, f'{LBL_NDVI}={p95:.3f}', va='center',
             fontsize=9, color='darkred', transform=cbar.ax.transAxes)
mask_p99_zoom = ndvi_diff_zoom > p99
ov_p99 = np.zeros((*ndvi_diff_zoom.shape, 4), dtype=float)
ov_p99[mask_p99_zoom] = [0.85, 0.05, 0.05, 0.6]
ax.imshow(ov_p99)
ax.set_title(f'ΔNDVI crudo — {LBL_NDVI}\n{np.sum(mask_p99_zoom):,} px sin filtrar',
             fontsize=12, fontweight='bold')
ax.axis('off')
ax.legend(handles=[Patch(facecolor='red', alpha=0.6,
          label=f'{LBL_NDVI} crudo: {np.sum(mask_p99_zoom):,} px')],
          loc='lower right', fontsize=10, framealpha=0.9)

# Panel 4: Candidatos tras pipeline
ax = axes[1, 1]
ax.imshow(slope_zoom, cmap='terrain',
          vmin=0, vmax=np.nanpercentile(slope_resampled, 99), alpha=0.55)
ax.imshow(ndvi_diff_zoom, cmap='RdYlGn_r', vmin=-vmax_z, vmax=vmax_z, alpha=0.35)
ov_desc = np.zeros((*ndvi_diff_zoom.shape, 4), dtype=float)
ov_desc[mask_p99_zoom & ~mask_filtros_zoom] = [0.4, 0.4, 0.4, 0.5]
ax.imshow(ov_desc)
ov_final = np.zeros((*mask_filtros_zoom.shape, 4), dtype=float)
ov_final[mask_filtros_zoom] = [0.85, 0.05, 0.05, 0.95]
ax.imshow(ov_final)
reduccion_zoom = np.sum(mask_p99_zoom) - np.sum(mask_filtros_zoom)
pct_zoom       = 100 * reduccion_zoom / max(np.sum(mask_p99_zoom), 1)
ax.set_title(
    f'Candidatos tras pipeline\n'
    f'{np.sum(mask_filtros_zoom):,} px retenidos  |  −{reduccion_zoom:,} removidos ({pct_zoom:.1f}%)',
    fontsize=12, fontweight='bold'
)
ax.axis('off')
ax.legend(handles=[
    Patch(facecolor='red',  alpha=0.95, label=f'Candidatos: {np.sum(mask_filtros_zoom):,} px'),
    Patch(facecolor='gray', alpha=0.55, label=f'Descartados: {reduccion_zoom:,} px'),
], loc='lower right', fontsize=10, framealpha=0.9)

plt.tight_layout()
nombre_archivo = NOMBRE_AREA.lower().replace(' ', '_')
plt.savefig(f'/tmp/{nombre_archivo}_zoom_completo.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"✓ Guardado en /tmp/{nombre_archivo}_zoom_completo.png")

## (Verificación de tamaño de celda)

In [ ]:
# ============================================================
# Diagnóstico RGB — Zona de acercamiento
# ============================================================

# Redefinir crop_rgb con argumentos explícitos
def crop_rgb_diag(rgb_arr, py_min, py_max, px_min, px_max, h_ref, w_ref):
    h_rgb, w_rgb = rgb_arr.shape[:2]
    return rgb_arr[
        int(py_min * h_rgb / h_ref):int(py_max * h_rgb / h_ref),
        int(px_min * w_rgb / w_ref):int(px_max * w_rgb / w_ref),
        :3
    ]

print(f"Shape RGB pre: {rgb_pre_raw.shape}")
print(f"Dtype: {rgb_pre_raw.dtype}")
print(f"Min: {rgb_pre_raw.min():.3f}  Max: {rgb_pre_raw.max():.3f}  Media: {rgb_pre_raw.mean():.3f}")

# Recortar zona de zoom
rgb_pre_zoom  = crop_rgb_diag(rgb_pre_raw,  py_min, py_max, px_min, px_max, h, w)
rgb_post_zoom = crop_rgb_diag(rgb_post_raw, py_min, py_max, px_min, px_max, h, w)

print(f"\nZoom pre  — shape: {rgb_pre_zoom.shape}  min: {rgb_pre_zoom.min():.3f}  max: {rgb_pre_zoom.max():.3f}")
print(f"Zoom post — shape: {rgb_post_zoom.shape}  min: {rgb_post_zoom.min():.3f}  max: {rgb_post_zoom.max():.3f}")

if rgb_pre_zoom.size == 0 or rgb_post_zoom.size == 0:
    raise ValueError(
        f"Zoom fuera del AOI — array vacío.\n"
        f"px: [{px_min}:{px_max}]  py: [{py_min}:{py_max}]\n"
        f"AOI bounds: {bounds}"
    )

# Normalizar para display si ya está en [0,1]
def norm_zoom(arr):
    out = np.zeros_like(arr, dtype=float)
    for c in range(3):
        ch = arr[..., c].astype(float)
        p2, p98 = np.percentile(ch, 2), np.percentile(ch, 98)
        out[..., c] = np.clip((ch - p2) / (p98 - p2 + 1e-10), 0, 1)
    return out

# Preparar versiones para display
pre_disp  = norm_zoom(rgb_pre_zoom)  if rgb_pre_raw.max()  > 1 else rgb_pre_zoom
post_disp = norm_zoom(rgb_post_zoom) if rgb_post_raw.max() > 1 else rgb_post_zoom
pre_norm  = norm_zoom(rgb_pre_zoom)
post_norm = norm_zoom(rgb_post_zoom)

# === Figura 2×2 =============================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Diagnóstico RGB — Zona de acercamiento\n{NOMBRE_AREA}',
             fontsize=13, fontweight='bold')

axes[0, 0].imshow(pre_disp)
axes[0, 0].set_title('Pre — original', fontsize=11, fontweight='bold')
axes[0, 0].axis('off')

axes[0, 1].imshow(post_disp)
axes[0, 1].set_title('Post — original', fontsize=11, fontweight='bold')
axes[0, 1].axis('off')

axes[1, 0].imshow(pre_norm)
axes[1, 0].set_title('Pre — normalizado P2-P98', fontsize=11, fontweight='bold')
axes[1, 0].axis('off')

axes[1, 1].imshow(post_norm)
axes[1, 1].set_title('Post — normalizado P2-P98', fontsize=11, fontweight='bold')
axes[1, 1].axis('off')

plt.tight_layout()
plt.savefig(f'/tmp/{NOMBRE_AREA.lower().replace(" ","_")}_diagnostico_rgb.png',
            dpi=150, bbox_inches='tight')
plt.show()

# Comparar arrays
diff_rgb = np.abs(rgb_pre_raw.astype(float) - rgb_post_raw.astype(float))
print(f"\n¿Arrays RGB idénticos?: {np.array_equal(rgb_pre_raw, rgb_post_raw)}")
print(f"Diferencia máxima entre pre y post: {diff_rgb.max():.4f}")
print(f"Diferencia media entre pre y post:  {diff_rgb.mean():.4f}")

In [ ]:
# Calcular resolución efectiva real de lo que estamos descargando
lon_min_aoi, lat_min_aoi, lon_max_aoi, lat_max_aoi = bounds

width_km  = (lon_max_aoi - lon_min_aoi) * 111 * np.cos(np.radians((lat_min_aoi + lat_max_aoi) / 2))
height_km = (lat_max_aoi - lat_min_aoi) * 111

h_arr, w_arr = rgb_pre_raw.shape[:2]

res_x = (width_km  * 1000) / w_arr
res_y = (height_km * 1000) / h_arr

print(f"Extensión del AOI:  {width_km:.1f} km × {height_km:.1f} km")
print(f"Tamaño del array:   {w_arr} × {h_arr} px")
print(f"Resolución efectiva: {res_x:.0f} m/px (X)  ×  {res_y:.0f} m/px (Y)")
print(f"\nPara obtener 10m/px necesitarías: {int(width_km*1000/10)} × {int(height_km*1000/10)} px")
print(f"Para obtener 30m/px necesitarías: {int(width_km*1000/30)} × {int(height_km*1000/30)} px")

## 12 - Filtro de Slope Units con Susceptibilidad muy alta, alta y media

In [ ]:
# ============================================================
# 12 - Máscara de Slope Units por Susceptibilidad
# Solo se analizan zonas: Muy Alta, Alta, Media
# Depende de: ndvi_diff_arr, ladera_mask, bounds, slope_resampled
# ============================================================

import geopandas as gpd
from rasterio.features import rasterize
from rasterio.transform import from_bounds
from matplotlib.patches import Patch

# ─── 1. Cargar shapefile de slope units ─────────────────────
shp_susc = "/home/ecastilloc/Descargas/WP.1.2.5/4_Febrero2026/data/Susceptibilidad/SlopeUnits_Cisneros.shp"
gdf_susc = gpd.read_file(shp_susc).to_crs(epsg=4326)

print(f"✓ Slope Units cargadas: {len(gdf_susc)} polígonos")
print(f"  Valores únicos en Nivel_Susc: {sorted(gdf_susc['Nivel_Susc'].unique())}")

# ─── 2. Filtrar categorías activas ──────────────────────────
categorias_activas = ['Muy Alto', 'Alto', 'Medio']

# Correcto
gdf_susc['Nivel_Susc'] = gdf_susc['Nivel_Susc'].str.strip()
gdf_activas = gdf_susc[gdf_susc['Nivel_Susc'].isin(['Muy Alto', 'Alto', 'Medio'])]

gdf_descartadas = gdf_susc[~gdf_susc['Nivel_Susc'].str.lower().str.strip().isin(categorias_activas)]

print(f"  Polígonos activos   (muy alta/alta/media): {len(gdf_activas)}")
print(f"  Polígonos excluidos (baja/muy baja):        {len(gdf_descartadas)}")

# ─── 3. Rasterizar al mismo grid que ndvi_diff_arr ──────────
h_ndvi, w_ndvi = ndvi_diff_arr.shape
lon_min_aoi, lat_min_aoi, lon_max_aoi, lat_max_aoi = bounds

transform_ndvi = from_bounds(
    lon_min_aoi, lat_min_aoi,
    lon_max_aoi, lat_max_aoi,
    w_ndvi, h_ndvi
)

mask_susc = rasterize(
    [(g, 1) for g in gdf_activas.geometry],
    out_shape=(h_ndvi, w_ndvi),
    transform=transform_ndvi,
    fill=0,
    dtype='uint8',
    all_touched=True
).astype(bool)

print(f"\n✓ Máscara rasterizada: {mask_susc.shape}")
print(f"  Píxeles activos:  {np.sum(mask_susc):,} ({100*np.sum(mask_susc)/mask_susc.size:.1f}%)")
print(f"  Píxeles excluidos: {np.sum(~mask_susc):,} ({100*np.sum(~mask_susc)/mask_susc.size:.1f}%)")

# ─── 4. Máscara combinada: pendiente + susceptibilidad ──────
# NDVI activo + BSI + Pendiente + Curvatura + Susceptibilidad
mask_final      = mask_susc & mask_p99_bsi_slope_curv
ndvi_diff_final = np.where(mask_final, ndvi_diff_arr, np.nan)

valid_final = ndvi_diff_final[~np.isnan(ndvi_diff_final)].ravel()
p90_f = np.percentile(valid_final, 90)
p95_f = np.percentile(valid_final, 95)
p99_f = np.percentile(valid_final, 99)

# Umbral ACTIVO de la segunda etapa (define los objetos del DBSCAN)
if PCT_NDVI_FINAL > 0:
    p_final = float(np.percentile(valid_final, PCT_NDVI_FINAL))
else:
    p_final = float(np.nanmin(valid_final)) - 1e-9   # desactivado: pasan todos

print(f"Umbrales ΔNDVI (todos los filtros aplicados):")
print(f"  P90: {p90_f:.4f}")
print(f"  P95: {p95_f:.4f}")
print(f"  P99: {p99_f:.4f}")
print(f"  ACTIVO P{PCT_NDVI_FINAL}: {p_final:.4f}")
print(f"  Píxeles válidos: {np.sum(~np.isnan(ndvi_diff_final)):,}")

# Verificar reducción acumulativa
print(f"\nReducción por etapa:")
print(f"  NDVI P{PCT_NDVI} original:          {np.sum(mask_p95):,} px")
print(f"  + BSI:                     {np.sum(mask_p99_bsi):,} px")
print(f"  + Pendiente + Curvatura:   {np.sum(mask_p99_bsi_slope_curv):,} px")
print(f"  + Susceptibilidad (final): {np.sum(mask_final):,} px")

# ─── 5. Visualización ───────────────────────────────────────
color_susc = {
    'muy alta': '#d73027',
    'alta':     '#fc8d59',
    'media':    '#fee090',
    'baja':     '#91bfdb',
    'muy baja': '#4575b4',
}

fig, axes = plt.subplots(1, 3, figsize=(21, 8))
fig.suptitle(
    'Filtro de Slope Units por Susceptibilidad\n'
    'Solo se analizan zonas: Susceptibilidad Media, Alta y Muy Alta',
    fontsize=14, fontweight='bold'
)

# --- Panel 1: Mapa de susceptibilidad ---
ax = axes[0]
ax.imshow(ndvi_diff_arr, cmap='Greys', alpha=0.3,
          vmin=np.nanpercentile(ndvi_diff_arr, 2),
          vmax=np.nanpercentile(ndvi_diff_arr, 98))

for nivel, color in color_susc.items():
    subset = gdf_susc[gdf_susc['Nivel_Susc'].str.lower().str.strip() == nivel]
    if not subset.empty:
        sub_mask = rasterize(
            [(g, 1) for g in subset.geometry],
            out_shape=(h_ndvi, w_ndvi),
            transform=transform_ndvi,
            fill=0, dtype='uint8', all_touched=True
        ).astype(bool)
        overlay = np.zeros((*sub_mask.shape, 4), dtype=float)
        r, g_c, b = tuple(int(color.lstrip('#')[i:i+2], 16)/255 for i in (0, 2, 4))
        overlay[sub_mask] = [r, g_c, b, 0.6]
        ax.imshow(overlay)

ax.set_title('Mapa de Susceptibilidad\n(Slope Units)', fontsize=11, fontweight='bold')
ax.axis('off')
ax.legend(handles=[Patch(facecolor=c, alpha=0.7, label=n.capitalize())
                   for n, c in color_susc.items()],
          loc='lower right', fontsize=9, framealpha=0.85, title='Susceptibilidad')

# --- Panel 2: ΔNDVI filtrado --- (corregir título)
ax = axes[1]
vmax = np.nanpercentile(np.abs(ndvi_diff_final), 98) if len(valid_final) > 0 else 0.3
im   = ax.imshow(ndvi_diff_final, cmap='RdYlGn_r', vmin=-vmax, vmax=vmax)
ax.set_title('ΔNDVI filtrado — Pipeline completo\n'
             f'NDVI P{PCT_NDVI} + BSI P{PCT_BSI} + Pendiente + Curvatura + Susceptibilidad',
             fontsize=11, fontweight='bold')
ax.axis('off')
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='ΔNDVI')

# --- Panel 3: Pendiente + Candidatos (umbral final) ---
ax = axes[2]

# Fondo 1: pendiente
im_slope = ax.imshow(slope_resampled, cmap='terrain',
                     vmin=0, vmax=np.nanpercentile(slope_resampled, 99), alpha=0.75)

# Fondo 2: ΔNDVI filtrado semitransparente
vmax_p = np.nanpercentile(np.abs(ndvi_diff_final), 98)
ax.imshow(ndvi_diff_final, cmap='RdYlGn_r',
          vmin=-vmax_p, vmax=vmax_p, alpha=0.30)

# Candidatos del umbral final en rojo sólido
mask_p99_final  = ndvi_diff_final > p_final
overlay_p99     = np.zeros((*mask_p99_final.shape, 4), dtype=float)
overlay_p99[mask_p99_final] = (0.85, 0.05, 0.05, 0.92)
ax.imshow(overlay_p99)

ax.set_title(
    f'Pendiente + Candidatos {LBL_NDVI_FINAL} (pipeline completo)\n'
    f'ΔNDVI > {p_final:.3f} | {np.sum(mask_p99_final):,} píxeles detectados',
    fontsize=11, fontweight='bold'
)
ax.axis('off')

# Colorbar de pendiente
from mpl_toolkits.axes_grid1 import make_axes_locatable
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="4%", pad=0.05)
plt.colorbar(im_slope, cax=cax, label='Pendiente (°)')

ax.legend(handles=[
    Patch(facecolor='darkred', alpha=0.9, label=f'{LBL_NDVI_FINAL}: ΔNDVI > {p_final:.3f}'),
], loc='lower right', fontsize=10, framealpha=0.85)

plt.tight_layout()
plt.savefig('/tmp/susceptibilidad_final.png', dpi=200, bbox_inches='tight')
plt.show()

print(f"\n✓ Guardado en /tmp/susceptibilidad_final.png")
print(f"\nResumen de filtros:")
print(f"  Pendiente > 15° (ladera_mask): {np.sum(ladera_mask):,} px")
print(f"  Susceptibilidad M/A/MA:        {np.sum(mask_susc):,} px")
print(f"  Filtro combinado (mask_final): {np.sum(mask_final):,} px")
print(f"  Candidatos {LBL_NDVI_FINAL}:              {np.sum(mask_p99_final):,} px")

In [ ]:
# === Mapa de Susceptibilidad + Candidatos (umbral final) ===

fig, ax = plt.subplots(1, 1, figsize=(14, 12))
fig.suptitle(
    f'Susceptibilidad por Slope Units + Candidatos a Movimiento en Masa ({LBL_NDVI_FINAL})\n'
    'Àrea de estudio',
    fontsize=15, fontweight='bold'
)

# ─── 1. Fondo: NDVI diff en grises ───────────────────────────
ax.imshow(ndvi_diff_arr, cmap='Greys', alpha=0.5,
          vmin=np.nanpercentile(ndvi_diff_arr, 2),
          vmax=np.nanpercentile(ndvi_diff_arr, 98))

# ─── 2. Slope Units por categoría (transparencia alta) ───────
color_susc = {
    'Muy Alto': '#d73027',
    'Alto':     '#fc8d59',
    'Medio':    '#fee090',
    'Bajo':     '#91bfdb',
    'Muy Bajo': '#4575b4',
}

for nivel, color in color_susc.items():
    subset = gdf_susc[gdf_susc['Nivel_Susc'] == nivel]
    if not subset.empty:
        sub_mask = rasterize(
            [(g, 1) for g in subset.geometry],
            out_shape=(h_ndvi, w_ndvi),
            transform=transform_ndvi,
            fill=0, dtype='uint8', all_touched=True
        ).astype(bool)
        overlay = np.zeros((*sub_mask.shape, 4), dtype=float)
        r, g_c, b = tuple(int(color.lstrip('#')[i:i+2], 16)/255 for i in (0, 2, 4))
        overlay[sub_mask] = [r, g_c, b, 0.35]  # transparencia alta
        ax.imshow(overlay)

# ─── 3. Candidatos del umbral final en rojo intenso ──────────
mask_p99_final = ndvi_diff_final > p_final
overlay_p99    = np.zeros((*mask_p99_final.shape, 4), dtype=float)
overlay_p99[mask_p99_final] = [0.85, 0.05, 0.05, 0.95]  # rojo sólido
ax.imshow(overlay_p99)

ax.axis('off')

# ─── 4. Leyenda combinada ─────────────────────────────────────
from matplotlib.patches import Patch

legend_elements = []
for nivel, color in color_susc.items():
    count = len(gdf_susc[gdf_susc['Nivel_Susc'] == nivel])
    legend_elements.append(
        Patch(facecolor=color, edgecolor='gray', linewidth=0.5,
              alpha=0.5, label=f'Susc. {nivel}  ({count} SU)')
    )
legend_elements.append(
    Patch(facecolor='#d40000', alpha=0.95,
          label=f'Candidatos {LBL_NDVI_FINAL}  (ΔNDVI > {p_final:.3f})\n{np.sum(mask_p99_final):,} píxeles')
)

ax.legend(handles=legend_elements, loc='lower right', fontsize=11,
          framealpha=0.92, title='Leyenda', title_fontsize=12,
          edgecolor='gray')

plt.tight_layout()
plt.savefig(f'/tmp/mapa_susceptibilidad_P{PCT_NDVI_FINAL}.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"✓ Guardado en /tmp/mapa_susceptibilidad_P{PCT_NDVI_FINAL}.png")
print(f"  Candidatos {LBL_NDVI_FINAL}: {np.sum(mask_p99_final):,} píxeles")

In [ ]:
# === Panel 2 ampliado: ΔNDVI filtrado ===

fig, ax = plt.subplots(1, 1, figsize=(14, 12))
fig.suptitle(
    'ΔNDVI filtrado — Pipeline completo\n'
    f'NDVI P{PCT_NDVI} + BSI P{PCT_BSI} + Pendiente > {SLOPE_THRESHOLD}° + Curvatura + Susc. Media/Alta/Muy Alta\n'
    'Àrea de estudio',
    fontsize=15, fontweight='bold'
)

vmax = np.nanpercentile(np.abs(ndvi_diff_final), 98)
im   = ax.imshow(ndvi_diff_final, cmap='RdYlGn_r', vmin=-vmax, vmax=vmax)
ax.axis('off')

cbar = plt.colorbar(im, ax=ax, fraction=0.035, pad=0.03, shrink=0.7)
cbar.set_label('ΔNDVI (Pre − Post)', fontsize=12)
cbar.ax.tick_params(labelsize=11)

# Líneas de umbral en la colorbar
for p, val, color, ls in zip(
    [PCT_NDVI_FINAL, 90,       95,      99],
    [p_final,        p90_f,    p95_f,   p99_f],
    ['blue',         'orange', 'red',   'darkred'],
    ['-',            '--',     '-.',    ':']
):
    cbar.ax.axhline(
        (val + vmax) / (2 * vmax),  # normalizar a [0,1] para posición en colorbar
        color=color, linewidth=2, linestyle=ls,
        label=f'P{p}: {val:.3f}'
    )

ax.legend(loc='lower right', fontsize=11, framealpha=0.92,
          title='Umbrales ΔNDVI', title_fontsize=12, edgecolor='gray')

plt.tight_layout()
plt.savefig('/tmp/ndvi_diff_filtrado_ampliado.png', dpi=200, bbox_inches='tight')
plt.show()
print(f"✓ Guardado en /tmp/ndvi_diff_filtrado_ampliado.png")
print(f"  Rango ΔNDVI: {np.nanmin(ndvi_diff_final):.3f} → {np.nanmax(ndvi_diff_final):.3f}")
print(f"  P90: {p90_f:.4f} | P95: {p95_f:.4f} | P99: {p99_f:.4f} | ACTIVO P{PCT_NDVI_FINAL}: {p_final:.4f}")

# (Filtrado completo = NDVI diff + BSI + Slope + Curvature + High susceptibility)

In [ ]:
# === Pipeline completo — reducción acumulativa con susceptibilidad ===

from matplotlib.patches import Patch

fig = plt.figure(figsize=(24, 28))
fig.suptitle(
    'Pipeline de filtros — Reducción acumulativa de candidatos\n'
    'Àrea de estudio',
    fontsize=17, fontweight='bold', y=1.01
)

# Layout: 3 filas × 2 columnas para los 5 mapas + 1 gráfico
gs = fig.add_gridspec(3, 2, hspace=0.35, wspace=0.15)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, 0])
ax4 = fig.add_subplot(gs[1, 1])
ax5 = fig.add_subplot(gs[2, 0])
ax6 = fig.add_subplot(gs[2, 1])

axes_flat = [ax1, ax2, ax3, ax4, ax5, ax6]

# ─── Etapas acumulativas completas ───────────────────────────
# 1. Primera etapa — usar mask_p99
etapas = [
    (mask_p99,
     f'Filtro 1 — NDVI {LBL_NDVI}',
     f'ΔNDVI > {p99:.3f}\n{np.sum(mask_p99):,} px  (100%)',
     ndvi_diff_arr, 'RdYlGn_r'),

    (mask_p99_bsi,
     f'Filtro 2 — + BSI {LBL_BSI}',
     f'ΔBSI > {BSI_ACT:.3f}\n{np.sum(mask_p99_bsi):,} px  '
     f'(−{np.sum(mask_p99)-np.sum(mask_p99_bsi):,} removidos)',
     bsi_diff_arr, 'YlOrBr'),

    (mask_p99_bsi & ladera_mask,
     'Filtro 3 — + Pendiente > 15°',
     f'{np.sum(mask_p99_bsi & ladera_mask):,} px  '
     f'(−{np.sum(mask_p99_bsi)-np.sum(mask_p99_bsi & ladera_mask):,} removidos)',
     slope_resampled, 'terrain'),

    (mask_p99_bsi_slope_curv,
     'Filtro 4 — + Curvatura',
     f'Excluye 15% más convexo\n{np.sum(mask_p99_bsi_slope_curv):,} px  '
     f'(−{np.sum(mask_p99_bsi & ladera_mask)-np.sum(mask_p99_bsi_slope_curv):,} removidos)',
     curvatura_arr, 'RdBu_r'),

    (mask_final,
     'Filtro 5 — + Susceptibilidad',
     f'Susc. Media/Alta/Muy Alta\n{np.sum(mask_final):,} px  '
     f'(−{np.sum(mask_p99_bsi_slope_curv)-np.sum(mask_final):,} removidos)',
     None, None),  # fondo especial — se maneja abajo,
]

mascaras = [m for m, *_ in etapas]

# ─── Paneles 1-5: mapas por filtro ───────────────────────────
for i, (ax, (mask_e, titulo, subtitulo, fondo, fcmap)) in \
        enumerate(zip(axes_flat[:5], etapas)):

    # ── Fondo especial para panel 5: susceptibilidad ──────────
    if i == 4:
        ax.imshow(ndvi_diff_arr, cmap='Greys', alpha=0.3,
                  vmin=np.nanpercentile(ndvi_diff_arr, 2),
                  vmax=np.nanpercentile(ndvi_diff_arr, 98))
        color_susc_pipeline = {
            'Muy Alto': '#d73027', 'Alto': '#fc8d59',
            'Medio': '#fee090', 'Bajo': '#91bfdb', 'Muy Bajo': '#4575b4',
        }
        for nivel, color in color_susc_pipeline.items():
            subset = gdf_susc[gdf_susc['Nivel_Susc'] == nivel]
            if not subset.empty:
                sub_mask = rasterize(
                    [(g, 1) for g in subset.geometry],
                    out_shape=mask_e.shape,
                    transform=transform_ndvi,
                    fill=0, dtype='uint8', all_touched=True
                ).astype(bool)
                ov_s = np.zeros((*sub_mask.shape, 4), dtype=float)
                r, g_c, b = tuple(int(color.lstrip('#')[j:j+2], 16)/255 for j in (0,2,4))
                ov_s[sub_mask] = [r, g_c, b, 0.45]
                ax.imshow(ov_s)
    else:
        # Fondo normal para los demás paneles
        vmin_f = 0 if fcmap == 'terrain' else np.nanpercentile(fondo, 2)
        vmax_f = np.nanpercentile(fondo, 98)
        ax.imshow(fondo, cmap=fcmap, vmin=vmin_f, vmax=vmax_f, alpha=0.55)

    # ── Píxeles removidos por este filtro en gris (para todos los paneles) ──
    if i > 0:
        mask_prev = mascaras[i - 1]
        ov_desc = np.zeros((*mask_e.shape, 4), dtype=float)
        ov_desc[mask_prev & ~mask_e] = [0.4, 0.4, 0.4, 0.55]
        ax.imshow(ov_desc)

    # ── Píxeles retenidos en rojo (para todos los paneles) ──────────────────
    ov_ret = np.zeros((*mask_e.shape, 4), dtype=float)
    ov_ret[mask_e] = [0.85, 0.05, 0.05, 0.92]
    ax.imshow(ov_ret)

    # 2. Porcentaje en títulos — referencia a p99
    pct_orig = 100 * np.sum(mask_e) / np.sum(mask_p99)
    removidos = np.sum(mascaras[i-1] & ~mask_e) if i > 0 else 0

    ax.set_title(f'{titulo}\n{subtitulo}\n({pct_orig:.1f}% del {LBL_NDVI} original)',
                 fontsize=12, fontweight='bold', pad=8)
    ax.axis('off')
    ax.legend(handles=[
        Patch(facecolor='red',  alpha=0.9, label=f'Retenidos: {np.sum(mask_e):,}'),
        Patch(facecolor='gray', alpha=0.6, label=f'Removidos: {removidos:,}'),
    ], loc='lower right', fontsize=10, framealpha=0.9)

# ─── Panel 6: gráfico de barras resumen ──────────────────────
ax = axes_flat[5]
ax.set_facecolor('#f8f8f8')

# 3. Gráfico de barras — etiqueta correcta
nombres = [f'NDVI\n{LBL_NDVI}', f'+BSI\n{LBL_BSI}', '+Pendiente\n>15°',
           '+Curvatura\nP85', '+Suscept.\nM/A/MA']
valores = [np.sum(m) for m in mascaras]
colores = ['#d73027', '#fc8d59', '#4dac26', '#4575b4', '#762a83']

bars = ax.bar(nombres, valores, color=colores,
              edgecolor='white', linewidth=1.5, width=0.6)

for bar, val in zip(bars, valores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
            f'{val:,}', ha='center', va='bottom',
            fontsize=11, fontweight='bold')

for i in range(len(valores) - 1):
    removidos_i = valores[i] - valores[i+1]
    pct_i = 100 * removidos_i / valores[i]
    ax.annotate(
        f'−{removidos_i:,}\n({pct_i:.0f}%)',
        xy=(i + 0.72, valores[i+1] + (valores[i] - valores[i+1]) / 2),
        fontsize=10, color='gray', ha='left', va='center'
    )

ax.axhline(valores[-1], color='darkred', linewidth=1.5,
           linestyle='--', alpha=0.6)
ax.text(len(valores) - 0.5, valores[-1] + 30,
        f'Final: {valores[-1]:,} px\n'
        f'({100*valores[-1]/valores[0]:.1f}% del original)',
        fontsize=10, color='darkred', ha='center', va='bottom')


ax.set_ylabel('Número de píxeles', fontsize=12)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.tick_params(labelsize=11)
ax.grid(True, axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)


# ─── Tamaño de panel ajustado al aspecto real de la imagen ───
h_img, w_img = ndvi_diff_arr.shape
img_aspect   = w_img / h_img

PANEL_W = 13                       # ← sube/baja este número para escalar todo
PANEL_H = PANEL_W / img_aspect

fig = plt.figure(figsize=(2 * PANEL_W, 3 * PANEL_H))
fig.suptitle(
    'Pipeline de filtros — Reducción acumulativa de candidatos\n'
    'Àrea de estudio',
    fontsize=19, fontweight='bold', y=1.01
)

gs = fig.add_gridspec(3, 2, hspace=0.14, wspace=0.04)

plt.savefig('/tmp/venecia_pipeline_completo.png', dpi=180, bbox_inches='tight')
plt.show()

print("✓ Guardado en /tmp/venecia_pipeline_completo.png")
print(f"\nResumen pipeline completo:")
for (mask_e, titulo, *_), val in zip(etapas, valores):
    print(f"  {titulo:35s}: {val:,} px")
reduccion_total = valores[0] - valores[-1]
print(f"\n  Reducción total: {reduccion_total:,} px "
      f"({100*reduccion_total/valores[0]:.1f}% eliminados)")
print(f"  Candidatos finales: {valores[-1]:,} px "
      f"({100*valores[-1]/valores[0]:.1f}% retenidos)")

## 13 - Validación con Inventario de Deslizamientos
Comparación cuantitativa entre cada etapa del pipeline y el inventario real

In [ ]:
# ============================================================
# 16 - Validación con Inventario de Deslizamientos
# Métricas POR EVENTO (no por píxel):
# Un deslizamiento se considera detectado si al menos 1 píxel
# de la máscara cae dentro de su polígono
# ============================================================

import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Patch
from rasterio.features import rasterize
from rasterio.transform import from_bounds
from pyproj import Geod

# ─── 1. Cargar inventario ────────────────────────────────────
inv_path = "/home/ecastilloc/Descargas/WP.1.2.5/5_Marzo_2026/INVENTARIOS/8_SantoDomingo_2021/Inventario_SantoDomingo_2021.shp"

gdf_inv = gpd.read_file(inv_path).to_crs(epsg=4326)
print(f"✓ Inventario cargado: {len(gdf_inv)} deslizamientos")
print(f"  CRS: {gdf_inv.crs}")
print(f"  Bounds: {gdf_inv.total_bounds}")

# ─── 2. Filtrar por área mínima ──────────────────────────────
AREA_MIN_M2 = 200
geod = Geod(ellps="WGS84")

def calc_area_m2(geom):
    area, _ = geod.geometry_area_perimeter(geom)
    return abs(area)

gdf_inv["area_m2"] = gdf_inv.geometry.apply(calc_area_m2)
gdf_inv = gdf_inv[gdf_inv["area_m2"] >= AREA_MIN_M2].copy().reset_index(drop=True)

print(f"\n  Inventario válido (>= {AREA_MIN_M2} m²): {len(gdf_inv)} deslizamientos")
print(f"  Área mín: {gdf_inv['area_m2'].min():.0f} m²  |  "
      f"Mediana: {gdf_inv['area_m2'].median():.0f} m²  |  "
      f"Máx: {gdf_inv['area_m2'].max():.0f} m²")

# ─── 3. Rasterizar inventario (para mapas visuales) ──────────
h_ndvi, w_ndvi = ndvi_diff_arr.shape
lon_min_aoi, lat_min_aoi, lon_max_aoi, lat_max_aoi = bounds

transform_val = from_bounds(
    lon_min_aoi, lat_min_aoi,
    lon_max_aoi, lat_max_aoi,
    w_ndvi, h_ndvi
)

mask_inv = rasterize(
    [(geom, 1) for geom in gdf_inv.geometry],
    out_shape=(h_ndvi, w_ndvi),
    transform=transform_val,
    fill=0, dtype="uint8", all_touched=True
).astype(bool)

# Rasterizar cada deslizamiento individualmente (para contar por evento)
print("\nRasterizando deslizamientos individualmente...")
masks_individuales = []
for idx, row in gdf_inv.iterrows():
    m = rasterize(
        [(row.geometry, 1)],
        out_shape=(h_ndvi, w_ndvi),
        transform=transform_val,
        fill=0, dtype="uint8", all_touched=True
    ).astype(bool)
    masks_individuales.append(m)

print(f"  ✓ {len(masks_individuales)} máscaras individuales generadas")

# ─── 4. Función métricas POR EVENTO ──────────────────────────
def metricas_por_evento(mask_det, masks_ind, gdf_inventario):
    """
    Cuenta TP/FP/FN por deslizamiento, no por píxel.
    - TP: deslizamiento con al menos 1 píxel detectado
    - FN: deslizamiento sin ningún píxel detectado
    - FP: clusters detectados que no tocan ningún deslizamiento
          (aproximado: píxeles detectados fuera de todos los polígonos)
    """
    tp_ids = []
    fn_ids = []

    for i, m_ind in enumerate(masks_ind):
        # ¿Hay algún píxel de la detección dentro de este deslizamiento?
        if np.any(mask_det & m_ind):
            tp_ids.append(i)
        else:
            fn_ids.append(i)

    TP = len(tp_ids)
    FN = len(fn_ids)

    # FP: píxeles detectados fuera de TODOS los polígonos del inventario
    # (ruido espacial puro)
    FP_px = int(np.sum(mask_det & ~mask_inv))

    # Precisión por evento: de los N eventos detectados, TP/(TP+FN_detectados)
    # Usamos una aproximación: Precisión = TP / (TP + FP_eventos_aprox)
    # FP_eventos = clusters detectados que no tocan ningún evento
    # Como no tenemos clusters aquí, usamos ratio píxeles FP / área media evento
    area_media_px = np.sum(mask_inv) / max(len(masks_ind), 1)
    FP_eventos = int(FP_px / max(area_media_px, 1))

    P  = TP / (TP + FP_eventos + 1e-10)
    R  = TP / (TP + FN + 1e-10)
    F1 = 2 * P * R / (P + R + 1e-10)

    return {
        "TP": TP, "FN": FN, "FP_px": FP_px, "FP_eventos": FP_eventos,
        "Precision": round(P, 4), "Recall": round(R, 4), "F1": round(F1, 4),
        "tp_ids": tp_ids, "fn_ids": fn_ids,
        "n_total": len(masks_ind)
    }

# También mantener métricas por píxel para comparación
def metricas_por_pixel(mask_det, mask_real):
    TP = np.sum(mask_det & mask_real)
    FP = np.sum(mask_det & ~mask_real)
    FN = np.sum(~mask_det & mask_real)
    P  = TP / (TP + FP + 1e-10)
    R  = TP / (TP + FN + 1e-10)
    F1 = 2 * P * R / (P + R + 1e-10)
    return {"TP": int(TP), "FP": int(FP), "FN": int(FN),
            "Precision": round(P, 4), "Recall": round(R, 4), "F1": round(F1, 4)}

# ─── 5. Calcular métricas por etapa ──────────────────────────
# Etapas 1-N: percentiles de referencia de ΔNDVI (el activo va marcado con ★)
# ─── 5. Calcular métricas por etapa ──────────────────────────
# Percentiles de ΔNDVI incluidos en la validación (P99 excluido:
# es una alternativa al umbral activo, no una etapa del pipeline)
PCTS_VALID = [p for p in PCTS_REF if p != 99]

etapas_eval = [
    (f"NDVI P{p}" + (" ★" if p == PCT_NDVI else ""), masks_ref[p])
    for p in PCTS_VALID
] + [
    (f"NDVI {LBL_NDVI} + BSI {LBL_BSI}",      mask_p99_bsi),
    (f"+ Pendiente > {SLOPE_THRESHOLD}°",     mask_p99_bsi & ladera_mask),
    ("+ Curvatura",                           mask_p99_bsi_slope_curv),
    ("+ Susceptibilidad",                     mask_final),
]

print(f"\n{'─'*100}")
print(f"  MÉTRICAS POR EVENTO (n={len(gdf_inv)} deslizamientos)")
print(f"{'─'*100}")
print(f"  {'Etapa':25s}  {'Píxeles':>9}  {'TP':>5}  {'FN':>5}  {'Recall':>8}  {'Precisión':>10}  {'F1':>8}")
print(f"  {'─'*95}")

resultados_evento = []
for nombre, mask_e in etapas_eval:
    m = metricas_por_evento(mask_e, masks_individuales, gdf_inv)
    resultados_evento.append({"etapa": nombre, "pixeles": int(np.sum(mask_e)), **m})
    print(f"  {nombre:25s}  {np.sum(mask_e):>9,}  "
          f"{m['TP']:>5}  {m['FN']:>5}  "
          f"{m['Recall']:>8.3f}  {m['Precision']:>10.3f}  {m['F1']:>8.3f}  "
          f"  ({m['TP']}/{m['n_total']} eventos detectados)")

df_metricas = pd.DataFrame(resultados_evento)

# Calcular m_final para celdas posteriores
m_final = metricas_por_evento(mask_final, masks_individuales, gdf_inv)
m_p99   = metricas_por_evento(mask_p95,   masks_individuales, gdf_inv)

print(f"\n✓ m_final y m_p99 calculados por evento")
print(f"  Pipeline final: {m_final['TP']}/{m_final['n_total']} deslizamientos detectados "
      f"(Recall={m_final['Recall']:.3f}  F1={m_final['F1']:.3f})")

# ─── 6. Visualización ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
fig.suptitle(
    f"Validación por Evento — {NOMBRE_AREA}\n"
    f"Inventario: {len(gdf_inv)} deslizamientos (>= {AREA_MIN_M2} m²)",
    fontsize=14, fontweight="bold"
)

x      = np.arange(len(etapas_eval))
labels = [r["etapa"] for r in resultados_evento]
tp_v   = [r["TP"]     for r in resultados_evento]
fn_v   = [r["FN"]     for r in resultados_evento]
rec_v  = [r["Recall"] for r in resultados_evento]
f1_v   = [r["F1"]     for r in resultados_evento]

# Panel 1: TP y FN por evento (barras apiladas)
ax = axes[0]
ax.bar(x, tp_v, color="green",  alpha=0.85, label="TP — Detectados")
ax.bar(x, fn_v, bottom=tp_v, color="orange", alpha=0.75, label="FN — No detectados")
for i, (tp, fn) in enumerate(zip(tp_v, fn_v)):
    ax.text(i, tp/2,        str(tp), ha="center", va="center",
            fontsize=9, fontweight="bold", color="white")
    ax.text(i, tp + fn/2,   str(fn), ha="center", va="center",
            fontsize=9, fontweight="bold", color="black")
ax.axhline(len(gdf_inv), color="darkblue", linewidth=1.5,
           linestyle="--", label=f"Total inventario: {len(gdf_inv)}")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("Número de deslizamientos", fontsize=11)
ax.set_title("TP / FN por evento\n(independiente del tamaño del deslizamiento)",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, axis="y", alpha=0.3)
ax.spines[["top","right"]].set_visible(False)

# Panel 2: Recall y F1 por etapa
ax = axes[1]
ax.plot(x, rec_v, "s-", color="darkorange", linewidth=2.5,
        markersize=9, label="Recall (eventos)")
ax.plot(x, f1_v,  "^-", color="darkgreen",  linewidth=2.5,
        markersize=9, label="F1-score (eventos)")
# Agregar esta línea después de rec_v y f1_v
prec_v = [r["Precision"] for r in resultados_evento]

ax.plot(x, prec_v, "o-", color="steelblue", linewidth=2.5,
        markersize=9, label="Precisión (eventos)")

# Y en las anotaciones agregar precisión
for i, (r, f, p) in enumerate(zip(rec_v, f1_v, prec_v)):
    ax.text(i, r + 0.02, f"{r:.2f}", ha="center", fontsize=8, color="darkorange")
    ax.text(i, f - 0.06, f"{f:.2f}", ha="center", fontsize=8, color="darkgreen")
    ax.text(i, p + 0.02, f"{p:.2f}", ha="center", fontsize=8, color="steelblue")

best_idx = int(np.argmax(f1_v))
ax.axvline(best_idx, color="darkgreen", linewidth=1.5, linestyle="--", alpha=0.5)
ax.text(best_idx + 0.1, 0.05,
        f"Mejor F1\n{labels[best_idx]}\nF1={f1_v[best_idx]:.3f}",
        color="darkgreen", fontsize=9, va="bottom")

for i, (r, f) in enumerate(zip(rec_v, f1_v)):
    ax.text(i, r + 0.02, f"{r:.2f}", ha="center", fontsize=8, color="darkorange")
    ax.text(i, f - 0.06, f"{f:.2f}", ha="center", fontsize=8, color="darkgreen")

ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha="right", fontsize=9)
ax.set_ylim(0, 1.1)
ax.set_ylabel("Métrica", fontsize=11)
ax.set_title("Recall / Precisión / F1 por etapa\n(conteo por evento)",
             fontsize=11, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
nombre_archivo = NOMBRE_AREA.lower().replace(" ","_")
plt.savefig(f"/tmp/{nombre_archivo}_validacion_por_evento.png", dpi=180, bbox_inches="tight")
plt.show()

print(f"\n✓ Guardado en /tmp/{nombre_archivo}_validacion_por_evento.png")
mejor = df_metricas.loc[df_metricas["F1"].idxmax()]
print(f"\nEtapa con mejor F1 (por evento):")
print(f"  {mejor['etapa']:30s} → F1={mejor['F1']:.3f}  "
      f"Recall={mejor['Recall']:.3f}  "
      f"TP={mejor['TP']}/{mejor['n_total']}")


In [ ]:
# === Mapa comparativo — 5 etapas clave + Inventario + TP/FN final ===

from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec

fig = plt.figure(figsize=(16, 110))
fig.suptitle(
    f'Comparación de etapas del pipeline vs Inventario — {NOMBRE_AREA}\n'
    f'Inventario: {len(gdf_inv)} deslizamientos  |  '
    f'Pipeline final — Recall: {m_final["Recall"]:.3f}  |  '
    f'F1: {m_final["F1"]:.3f}  |  '
    f'TP: {m_final["TP"]}/{m_final["n_total"]} eventos',
    fontsize=15, fontweight='bold'
)

gs   = gridspec.GridSpec(7, 1, figure=fig, hspace=0.18)
axes = [fig.add_subplot(gs[i]) for i in range(7)]

fondo  = ndvi_diff_arr
vmin_f = np.nanpercentile(fondo, 2)
vmax_f = np.nanpercentile(fondo, 98)

# Calcular métricas de etapas clave
m_p90          = metricas_por_evento(mask_p90, masks_individuales, gdf_inv)
m_p99          = metricas_por_evento(mask_p95, masks_individuales, gdf_inv)
m_bsi_slope    = metricas_por_evento(mask_p99_bsi & ladera_mask, masks_individuales, gdf_inv)

# ─── Helper: dibujar overlay rojo ───────────────────────────
def draw_red(ax, mask):
    ov = np.zeros((*mask.shape, 4), dtype=float)
    ov[mask] = [0.9, 0.1, 0.1, 0.85]
    ax.imshow(ov)

def draw_blue(ax, mask):
    ov = np.zeros((*mask.shape, 4), dtype=float)
    ov[mask] = [0.0, 0.3, 0.9, 0.85]
    ax.imshow(ov)

# ─── Mapa 1: Inventario real ─────────────────────────────────
ax = axes[0]
ax.imshow(fondo, cmap='Greys', alpha=0.45, vmin=vmin_f, vmax=vmax_f)
draw_blue(ax, mask_inv)
ax.set_title(
    f'Inventario real\n'
    f'{len(gdf_inv)} deslizamientos  |  {np.sum(mask_inv):,} px',
    fontsize=14, fontweight='bold'
)
ax.axis('off')
ax.legend(handles=[
    Patch(facecolor='blue', alpha=0.85,
          label=f'Inventario: {len(gdf_inv)} eventos'),
], loc='lower right', fontsize=11, framealpha=0.92)

# ─── Mapa 2: NDVI P90 ────────────────────────────────────────
ax = axes[1]
ax.imshow(fondo, cmap='Greys', alpha=0.45, vmin=vmin_f, vmax=vmax_f)
draw_red(ax, mask_p90)
ax.set_title(
    f'NDVI P90\n'
    f'{np.sum(mask_p90):,} px  |  '
    f'TP: {m_p90["TP"]}/{m_p90["n_total"]} eventos  |  '
    f'Recall: {m_p90["Recall"]:.3f}  |  F1: {m_p90["F1"]:.3f}',
    fontsize=14, fontweight='bold'
)
ax.axis('off')
ax.legend(handles=[
    Patch(facecolor='red', alpha=0.85,
          label=f'P90: {np.sum(mask_p90):,} px  |  {m_p90["TP"]} eventos detectados'),
], loc='lower right', fontsize=11, framealpha=0.92)

# ─── Mapa 3: NDVI P_NDVI ────────────────────────────────────────
ax = axes[2]
ax.imshow(fondo, cmap='Greys', alpha=0.45, vmin=vmin_f, vmax=vmax_f)
draw_red(ax, mask_p95)
ax.set_title(
    f'NDVI {LBL_NDVI}\n'
    f'{np.sum(mask_p95):,} px  |  '
    f'TP: {m_p99["TP"]}/{m_p99["n_total"]} eventos  |  '
    f'Recall: {m_p99["Recall"]:.3f}  |  F1: {m_p99["F1"]:.3f}',
    fontsize=14, fontweight='bold'
)
ax.axis('off')
ax.legend(handles=[
    Patch(facecolor='red', alpha=0.85,
          label=f'{LBL_NDVI}: {np.sum(mask_p95):,} px  |  {m_p99["TP"]} eventos detectados'),
], loc='lower right', fontsize=11, framealpha=0.92)

# ─── Mapa 4: P_NDVI + BSI + Pendiente ───────────────────────────
mask_bsi_slope = mask_p99_bsi & ladera_mask
ax = axes[3]
ax.imshow(fondo, cmap='Greys', alpha=0.45, vmin=vmin_f, vmax=vmax_f)
draw_red(ax, mask_bsi_slope)
ax.set_title(
    f'NDVI {LBL_NDVI} + BSI + Pendiente > {SLOPE_THRESHOLD}°\n'
    f'{np.sum(mask_bsi_slope):,} px  |  '
    f'TP: {m_bsi_slope["TP"]}/{m_bsi_slope["n_total"]} eventos  |  '
    f'Recall: {m_bsi_slope["Recall"]:.3f}  |  F1: {m_bsi_slope["F1"]:.3f}',
    fontsize=14, fontweight='bold'
)
ax.axis('off')
ax.legend(handles=[
    Patch(facecolor='red', alpha=0.85,
          label=f'BSI+Slope: {np.sum(mask_bsi_slope):,} px  |  '
                f'{m_bsi_slope["TP"]} eventos detectados'),
], loc='lower right', fontsize=11, framealpha=0.92)

# ─── Mapa 5: Pipeline completo ───────────────────────────────
ax = axes[4]
ax.imshow(fondo, cmap='Greys', alpha=0.45, vmin=vmin_f, vmax=vmax_f)
draw_red(ax, mask_final)
ax.set_title(
    f'Pipeline completo (todos los filtros)\n'
    f'{LBL_NDVI} + BSI + Pendiente + Curvatura + Susceptibilidad\n'
    f'{np.sum(mask_final):,} px  |  '
    f'TP: {m_final["TP"]}/{m_final["n_total"]} eventos  |  '
    f'Recall: {m_final["Recall"]:.3f}  |  F1: {m_final["F1"]:.3f}',
    fontsize=14, fontweight='bold'
)
ax.axis('off')
ax.legend(handles=[
    Patch(facecolor='red', alpha=0.85,
          label=f'Pipeline: {np.sum(mask_final):,} px  |  '
                f'{m_final["TP"]} eventos detectados'),
], loc='lower right', fontsize=11, framealpha=0.92)

# ─── Mapa 6: Curva Recall y F1 ───────────────────────────────
ax = axes[5]
ax.set_facecolor('#f8f8f8')

etapas_curva = [
    ('NDVI P90',   mask_p90),
    (f'NDVI {LBL_NDVI}',   mask_p95),
    (f'NDVI {LBL_NDVI}',   mask_p99),
    (f'{LBL_NDVI}+BSI',    mask_p99_bsi),
    ('+Pendiente', mask_p99_bsi & ladera_mask),
    ('+Curvatura', mask_p99_bsi_slope_curv),
    ('+Suscept.',  mask_final),
]
x_c    = np.arange(len(etapas_curva))
lab_c  = [n for n, _ in etapas_curva]
rec_c  = [metricas_por_evento(m, masks_individuales, gdf_inv)['Recall'] for _, m in etapas_curva]
f1_c   = [metricas_por_evento(m, masks_individuales, gdf_inv)['F1']     for _, m in etapas_curva]
tp_c   = [metricas_por_evento(m, masks_individuales, gdf_inv)['TP']     for _, m in etapas_curva]

ax.plot(x_c, rec_c, 's-', color='darkorange', linewidth=2.5, markersize=9, label='Recall (eventos)')
ax.plot(x_c, f1_c,  '^-', color='darkgreen',  linewidth=2.5, markersize=9, label='F1 (eventos)')

best_idx = int(np.argmax(f1_c))
ax.axvline(best_idx, color='darkgreen', linewidth=1.5, linestyle='--', alpha=0.5)
ax.text(best_idx + 0.1, 0.05,
        f'Mejor F1\n{lab_c[best_idx]}\nF1={f1_c[best_idx]:.3f}',
        color='darkgreen', fontsize=10, va='bottom')

for i, (r, f, tp) in enumerate(zip(rec_c, f1_c, tp_c)):
    ax.text(i, r + 0.02, f'{r:.2f}\n({tp}ev)', ha='center', fontsize=8, color='darkorange')
    ax.text(i, f - 0.07, f'{f:.2f}',            ha='center', fontsize=8, color='darkgreen')

ax.set_xticks(x_c)
ax.set_xticklabels(lab_c, rotation=30, ha='right', fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Métrica por evento', fontsize=12)
ax.set_title('Recall y F1 por etapa vs Inventario\n'
             '(conteo por evento — independiente del tamaño del deslizamiento)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

# ─── Mapa 7: TP / FN del pipeline final ──────────────────────
ax = axes[6]
ax.imshow(fondo, cmap='Greys', alpha=0.35, vmin=vmin_f, vmax=vmax_f)

# FP en rojo tenue (píxeles detectados fuera del inventario)
ov_fp = np.zeros((*mask_final.shape, 4), dtype=float)
ov_fp[mask_final & ~mask_inv] = [0.9, 0.1, 0.1, 0.45]
ax.imshow(ov_fp)

# FN: deslizamientos NO detectados → naranja
fn_mask_final = np.zeros(mask_inv.shape, dtype=bool)
for i in m_final['fn_ids']:
    fn_mask_final |= masks_individuales[i]
ov_fn = np.zeros((*mask_inv.shape, 4), dtype=float)
ov_fn[fn_mask_final] = [1.0, 0.5, 0.0, 0.90]
ax.imshow(ov_fn)

# TP: deslizamientos detectados → amarillo brillante
tp_mask_final = np.zeros(mask_inv.shape, dtype=bool)
for i in m_final['tp_ids']:
    tp_mask_final |= masks_individuales[i]
ov_tp = np.zeros((*mask_final.shape, 4), dtype=float)
ov_tp[tp_mask_final & mask_final] = [1.0, 1.0, 0.0, 0.98]
ax.imshow(ov_tp)

ax.set_title(
    f'Aciertos y desaciertos — Pipeline completo vs Inventario\n'
    f'TP: {m_final["TP"]} eventos detectados  |  '
    f'FN: {m_final["FN"]} eventos NO detectados  |  '
    f'FP: {m_final["FP_px"]:,} px fuera del inventario\n'
    f'Recall: {m_final["Recall"]:.3f}  |  F1: {m_final["F1"]:.3f}',
    fontsize=14, fontweight='bold'
)
ax.axis('off')
ax.legend(handles=[
    Patch(facecolor='yellow', alpha=0.98,
          label=f'TP — Detectados: {m_final["TP"]} eventos'),
    Patch(facecolor='orange', alpha=0.90,
          label=f'FN — No detectados: {m_final["FN"]} eventos'),
    Patch(facecolor='red',    alpha=0.50,
          label=f'FP — Ruido: {m_final["FP_px"]:,} px fuera inventario'),
], loc='lower right', fontsize=11, framealpha=0.92)

plt.tight_layout()
nombre_archivo = NOMBRE_AREA.lower().replace(' ', '_')
plt.savefig(f'/tmp/{nombre_archivo}_comparacion_etapas_clave.png', dpi=180, bbox_inches='tight')
plt.show()

print(f"✓ Guardado en /tmp/{nombre_archivo}_comparacion_etapas_clave.png")
print(f"\nResumen por etapas clave (métricas por evento):")
print(f"  {'Etapa':30s}  {'Píxeles':>10}  {'TP':>5}  {'FN':>5}  {'Recall':>8}  {'F1':>8}")
print(f"  {'─'*75}")
for nombre, mask_e, m_e in [
    ('Inventario',              mask_inv,       None),
    ('NDVI P90',                mask_p90,       m_p90),
    (f'NDVI {LBL_NDVI}',                mask_p95,       m_p99),
    (f'{LBL_NDVI} + BSI + Pendiente',   mask_bsi_slope, m_bsi_slope),
    ('Pipeline completo',       mask_final,     m_final),
]:
    if m_e is None:
        print(f"  {'Inventario':30s}  {np.sum(mask_inv):>10,}  "
              f"{len(gdf_inv):>5}  {'—':>5}  {'—':>8}  {'—':>8}")
    else:
        print(f"  {nombre:30s}  {np.sum(mask_e):>10,}  "
              f"{m_e['TP']:>5}  {m_e['FN']:>5}  "
              f"{m_e['Recall']:>8.3f}  {m_e['F1']:>8.3f}")

In [ ]:
# === Mapa comparativo — 5 etapas clave + Inventario + TP/FN final ===

from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker

fig = plt.figure(figsize=(16, 110))
fig.suptitle(
    f'Comparación de etapas del pipeline vs Inventario — {NOMBRE_AREA}\n'
    f'Inventario: {len(gdf_inv)} deslizamientos  |  '
    f'Pipeline final — Recall: {m_final["Recall"]:.3f}  |  '
    f'F1: {m_final["F1"]:.3f}  |  '
    f'TP: {m_final["TP"]}/{m_final["n_total"]} eventos',
    fontsize=15, fontweight='bold'
)

gs   = gridspec.GridSpec(7, 1, figure=fig, hspace=0.25)
axes = [fig.add_subplot(gs[i]) for i in range(7)]

fondo  = ndvi_diff_arr
vmin_f = np.nanpercentile(fondo, 2)
vmax_f = np.nanpercentile(fondo, 98)

# Bounds geográficos del AOI
lon_min_aoi, lat_min_aoi, lon_max_aoi, lat_max_aoi = bounds

# ─── Helper: grilla de coordenadas + escala ──────────────────
def add_grid_and_scale(ax, n_ticks_x=5, n_ticks_y=5):
    """
    Agrega grilla de coordenadas geográficas y barra de escala.
    Asume que el array cubre el extent (lon_min_aoi, lon_max_aoi,
    lat_min_aoi, lat_max_aoi) en píxeles (0,w) x (0,h).
    """
    h_px, w_px = fondo.shape

    # ── Ticks de longitud (eje X) ──────────────────────────
    lon_ticks = np.linspace(lon_min_aoi, lon_max_aoi, n_ticks_x)
    px_ticks_x = [(lon - lon_min_aoi) / (lon_max_aoi - lon_min_aoi) * w_px
                  for lon in lon_ticks]
    ax.set_xticks(px_ticks_x)
    ax.set_xticklabels([f'{lon:.3f}°' for lon in lon_ticks],
                       fontsize=11, color='gray')
    ax.tick_params(axis='x', length=5, color='gray')

    # ── Ticks de latitud (eje Y) ───────────────────────────
    lat_ticks = np.linspace(lat_min_aoi, lat_max_aoi, n_ticks_y)
    px_ticks_y = [(lat_max_aoi - lat) / (lat_max_aoi - lat_min_aoi) * h_px
                  for lat in lat_ticks]
    ax.set_yticks(px_ticks_y)
    ax.set_yticklabels([f'{lat:.3f}°' for lat in lat_ticks],
                       fontsize=11, color='gray')
    ax.tick_params(axis='y', length=5, color='gray')

    # ── Grilla ─────────────────────────────────────────────
    ax.grid(True, color='white', linewidth=0.6, alpha=0.5, linestyle='--')
    ax.set_axisbelow(False)

    # ── Barra de escala ────────────────────────────────────
    # Calcular cuántos píxeles equivalen a 1 km en X
    import math
    lat_mid   = (lat_min_aoi + lat_max_aoi) / 2
    km_per_deg_lon = 111.32 * math.cos(math.radians(lat_mid))
    px_per_km = (w_px / (lon_max_aoi - lon_min_aoi)) / km_per_deg_lon

    # Elegir longitud de escala redonda
    scale_km = 1
    for candidate in [0.5, 1, 2, 5, 10, 20]:
        if candidate * px_per_km < w_px * 0.25:
            scale_km = candidate

    scale_px = scale_km * px_per_km

    # Posición: esquina inferior izquierda
    x0 = w_px * 0.04
    y0 = h_px * 0.94
    bar_h = h_px * 0.012

    ax.add_patch(plt.Rectangle((x0, y0), scale_px, bar_h,
                                color='black', zorder=10))
    ax.add_patch(plt.Rectangle((x0 + scale_px/2, y0), scale_px/2, bar_h,
                                color='white', zorder=11))
    ax.text(x0, y0 - bar_h * 2.5, '0',
            fontsize=11, color='black', ha='center', zorder=12)   # ← 8 → 11
    ax.text(x0 + scale_px/2, y0 - bar_h * 2.5, f'{scale_km/2:.1f}',
            fontsize=11, color='black', ha='center', zorder=12)   # ← 8 → 11
    ax.text(x0 + scale_px, y0 - bar_h * 2.5, f'{scale_km} km',
            fontsize=11, color='black', ha='center', zorder=12)   # ← 8 → 11

# ─── Helper: overlay rojo / azul ─────────────────────────────
def draw_red(ax, mask):
    ov = np.zeros((*mask.shape, 4), dtype=float)
    ov[mask] = [0.9, 0.1, 0.1, 0.85]
    ax.imshow(ov)

def draw_blue(ax, mask):
    ov = np.zeros((*mask.shape, 4), dtype=float)
    ov[mask] = [0.0, 0.3, 0.9, 0.85]
    ax.imshow(ov)

# Calcular métricas etapas clave
m_p90       = metricas_por_evento(mask_p90, masks_individuales, gdf_inv)
m_p99       = metricas_por_evento(mask_p95, masks_individuales, gdf_inv)
mask_bsi_slope = mask_p99_bsi & ladera_mask
m_bsi_slope = metricas_por_evento(mask_bsi_slope, masks_individuales, gdf_inv)

# ─── Mapa 1: Inventario real ─────────────────────────────────
ax = axes[0]
ax.imshow(fondo, cmap='Greys', alpha=0.45, vmin=vmin_f, vmax=vmax_f)
draw_blue(ax, mask_inv)
add_grid_and_scale(ax)
ax.set_title(f'Inventario real\n{len(gdf_inv)} deslizamientos  |  {np.sum(mask_inv):,} px',
             fontsize=14, fontweight='bold')
ax.legend(handles=[
    Patch(facecolor='blue', alpha=0.85, label=f'Inventario: {len(gdf_inv)} eventos'),
], loc='lower right', fontsize=11, framealpha=0.92)

# ─── Mapa 2: NDVI P90 ────────────────────────────────────────
ax = axes[1]
ax.imshow(fondo, cmap='Greys', alpha=0.45, vmin=vmin_f, vmax=vmax_f)
draw_red(ax, mask_p90)
add_grid_and_scale(ax)
ax.set_title(
    f'NDVI P90\n'
    f'{np.sum(mask_p90):,} px  |  '
    f'TP: {m_p90["TP"]}/{m_p90["n_total"]} eventos  |  '
    f'Recall: {m_p90["Recall"]:.3f}  |  F1: {m_p90["F1"]:.3f}',
    fontsize=14, fontweight='bold'
)
ax.legend(handles=[
    Patch(facecolor='red', alpha=0.85,
          label=f'P90: {np.sum(mask_p90):,} px  |  {m_p90["TP"]} eventos'),
], loc='lower right', fontsize=11, framealpha=0.92)

# ─── Mapa 3: NDVI P_NDVI ────────────────────────────────────────
ax = axes[2]
ax.imshow(fondo, cmap='Greys', alpha=0.45, vmin=vmin_f, vmax=vmax_f)
draw_red(ax, mask_p95)
add_grid_and_scale(ax)
ax.set_title(
    f'NDVI {LBL_NDVI}\n'
    f'{np.sum(mask_p95):,} px  |  '
    f'TP: {m_p99["TP"]}/{m_p99["n_total"]} eventos  |  '
    f'Recall: {m_p99["Recall"]:.3f}  |  F1: {m_p99["F1"]:.3f}',
    fontsize=14, fontweight='bold'
)
ax.legend(handles=[
    Patch(facecolor='red', alpha=0.85,
          label=f'{LBL_NDVI}: {np.sum(mask_p95):,} px  |  {m_p99["TP"]} eventos'),
], loc='lower right', fontsize=11, framealpha=0.92)

# ─── Mapa 4: P_NDVI + BSI + Pendiente ───────────────────────────
ax = axes[3]
ax.imshow(fondo, cmap='Greys', alpha=0.45, vmin=vmin_f, vmax=vmax_f)
draw_red(ax, mask_bsi_slope)
add_grid_and_scale(ax)
ax.set_title(
    f'NDVI {LBL_NDVI} + BSI + Pendiente > {SLOPE_THRESHOLD}°\n'
    f'{np.sum(mask_bsi_slope):,} px  |  '
    f'TP: {m_bsi_slope["TP"]}/{m_bsi_slope["n_total"]} eventos  |  '
    f'Recall: {m_bsi_slope["Recall"]:.3f}  |  F1: {m_bsi_slope["F1"]:.3f}',
    fontsize=14, fontweight='bold'
)
ax.legend(handles=[
    Patch(facecolor='red', alpha=0.85,
          label=f'BSI+Slope: {np.sum(mask_bsi_slope):,} px  |  '
                f'{m_bsi_slope["TP"]} eventos'),
], loc='lower right', fontsize=11, framealpha=0.92)

# ─── Mapa 5: Pipeline completo ───────────────────────────────
ax = axes[4]
ax.imshow(fondo, cmap='Greys', alpha=0.45, vmin=vmin_f, vmax=vmax_f)
draw_red(ax, mask_final)
add_grid_and_scale(ax)
ax.set_title(
    f'Pipeline completo (todos los filtros)\n'
    f'{LBL_NDVI} + BSI + Pendiente + Curvatura + Susceptibilidad\n'
    f'{np.sum(mask_final):,} px  |  '
    f'TP: {m_final["TP"]}/{m_final["n_total"]} eventos  |  '
    f'Recall: {m_final["Recall"]:.3f}  |  F1: {m_final["F1"]:.3f}',
    fontsize=14, fontweight='bold'
)
ax.legend(handles=[
    Patch(facecolor='red', alpha=0.85,
          label=f'Pipeline: {np.sum(mask_final):,} px  |  '
                f'{m_final["TP"]} eventos'),
], loc='lower right', fontsize=11, framealpha=0.92)

# ─── Mapa 6: Curva Recall / F1 ───────────────────────────────
ax = axes[5]
ax.set_facecolor('#f8f8f8')

etapas_curva = [
    ('NDVI P90',   mask_p90),
    (f'NDVI {LBL_NDVI}',   mask_p95),
    (f'NDVI {LBL_NDVI}',   mask_p99),
    (f'{LBL_NDVI}+BSI',    mask_p99_bsi),
    ('+Pendiente', mask_p99_bsi & ladera_mask),
    ('+Curvatura', mask_p99_bsi_slope_curv),
    ('+Suscept.',  mask_final),
]
x_c   = np.arange(len(etapas_curva))
lab_c = [n for n, _ in etapas_curva]
rec_c = [metricas_por_evento(m, masks_individuales, gdf_inv)['Recall'] for _, m in etapas_curva]
f1_c  = [metricas_por_evento(m, masks_individuales, gdf_inv)['F1']     for _, m in etapas_curva]
tp_c  = [metricas_por_evento(m, masks_individuales, gdf_inv)['TP']     for _, m in etapas_curva]

ax.plot(x_c, rec_c, 's-', color='darkorange', linewidth=2.5, markersize=9, label='Recall (eventos)')
ax.plot(x_c, f1_c,  '^-', color='darkgreen',  linewidth=2.5, markersize=9, label='F1 (eventos)')

best_idx = int(np.argmax(f1_c))
ax.axvline(best_idx, color='darkgreen', linewidth=1.5, linestyle='--', alpha=0.5)
ax.text(best_idx + 0.1, 0.05,
        f'Mejor F1\n{lab_c[best_idx]}\nF1={f1_c[best_idx]:.3f}',
        color='darkgreen', fontsize=10, va='bottom')

for i, (r, f, tp) in enumerate(zip(rec_c, f1_c, tp_c)):
    ax.text(i, r + 0.02, f'{r:.2f}\n({tp}ev)', ha='center', fontsize=8, color='darkorange')
    ax.text(i, f - 0.07, f'{f:.2f}',            ha='center', fontsize=8, color='darkgreen')

ax.set_xticks(x_c)
ax.set_xticklabels(lab_c, rotation=30, ha='right', fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Métrica por evento', fontsize=12)
ax.set_title('Recall y F1 por etapa vs Inventario\n'
             '(conteo por evento — independiente del tamaño)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)

# ─── Mapa 7: TP / FN / FP pipeline final ─────────────────────
ax = axes[6]
ax.imshow(fondo, cmap='Greys', alpha=0.35, vmin=vmin_f, vmax=vmax_f)

ov_fp = np.zeros((*mask_final.shape, 4), dtype=float)
ov_fp[mask_final & ~mask_inv] = [0.9, 0.1, 0.1, 0.45]
ax.imshow(ov_fp)

fn_mask_final = np.zeros(mask_inv.shape, dtype=bool)
for i in m_final['fn_ids']:
    fn_mask_final |= masks_individuales[i]
ov_fn = np.zeros((*mask_inv.shape, 4), dtype=float)
ov_fn[fn_mask_final] = [1.0, 0.5, 0.0, 0.90]
ax.imshow(ov_fn)

tp_mask_final = np.zeros(mask_inv.shape, dtype=bool)
for i in m_final['tp_ids']:
    tp_mask_final |= masks_individuales[i]
ov_tp = np.zeros((*mask_final.shape, 4), dtype=float)
ov_tp[tp_mask_final & mask_final] = [1.0, 1.0, 0.0, 0.98]
ax.imshow(ov_tp)

add_grid_and_scale(ax)
ax.set_title(
    f'Aciertos y desaciertos — Pipeline completo vs Inventario\n'
    f'TP: {m_final["TP"]} eventos detectados  |  '
    f'FN: {m_final["FN"]} eventos NO detectados  |  '
    f'FP: {m_final["FP_px"]:,} px fuera inventario\n'
    f'Recall: {m_final["Recall"]:.3f}  |  F1: {m_final["F1"]:.3f}',
    fontsize=14, fontweight='bold'
)
ax.legend(handles=[
    Patch(facecolor='yellow', alpha=0.98,
          label=f'TP — Detectados: {m_final["TP"]} eventos'),
    Patch(facecolor='orange', alpha=0.90,
          label=f'FN — No detectados: {m_final["FN"]} eventos'),
    Patch(facecolor='red',    alpha=0.50,
          label=f'FP — Ruido: {m_final["FP_px"]:,} px'),
], loc='lower right', fontsize=11, framealpha=0.92)

plt.tight_layout()
nombre_archivo = NOMBRE_AREA.lower().replace(' ', '_')
plt.savefig(f'/tmp/{nombre_archivo}_comparacion_etapas_clave.png', dpi=180, bbox_inches='tight')
plt.show()

print(f"✓ Guardado en /tmp/{nombre_archivo}_comparacion_etapas_clave.png")
print(f"\nResumen por etapas clave (métricas por evento):")
print(f"  {'Etapa':30s}  {'Píxeles':>10}  {'TP':>5}  {'FN':>5}  {'Recall':>8}  {'F1':>8}")
print(f"  {'─'*75}")
for nombre, mask_e, m_e in [
    ('Inventario',            mask_inv,       None),
    ('NDVI P90',              mask_p90,       m_p90),
    (f'NDVI {LBL_NDVI}',              mask_p95,       m_p99),
    (f'{LBL_NDVI} + BSI + Pendiente', mask_bsi_slope, m_bsi_slope),
    ('Pipeline completo',     mask_final,     m_final),
]:
    if m_e is None:
        print(f"  {'Inventario':30s}  {np.sum(mask_inv):>10,}  "
              f"{len(gdf_inv):>5}  {'—':>5}  {'—':>8}  {'—':>8}")
    else:
        print(f"  {nombre:30s}  {np.sum(mask_e):>10,}  "
              f"{m_e['TP']:>5}  {m_e['FN']:>5}  "
              f"{m_e['Recall']:>8.3f}  {m_e['F1']:>8.3f}")

In [ ]:
# ============================================================
# 16b - Mapa de aciertos y desaciertos (validación por evento)
# Verde  = deslizamiento detectado (TP)
# Naranja= deslizamiento NO detectado (FN)
# Rojo   = píxeles detectados fuera del inventario (FP)
# Depende de: gdf_inv, masks_individuales, mask_inv, metricas_por_evento,
#             ndvi_diff_arr, transform_val, NOMBRE_AREA
# ============================================================

from matplotlib.patches import Patch
from matplotlib.lines import Line2D

# ─── Máscara a evaluar (cambiar aquí para comparar etapas) ───
MASK_EVAL   = mask_final
NOMBRE_EVAL = 'Pipeline completo'

m = metricas_por_evento(MASK_EVAL, masks_individuales, gdf_inv)

# ─── Máscaras derivadas ──────────────────────────────────────
det_dentro = MASK_EVAL & mask_inv          # píxeles detectados sobre inventario
det_fuera  = MASK_EVAL & ~mask_inv         # falsos positivos espaciales

mask_tp = np.zeros_like(mask_inv)
for i in m['tp_ids']:
    mask_tp |= masks_individuales[i]
mask_fn = np.zeros_like(mask_inv)
for i in m['fn_ids']:
    mask_fn |= masks_individuales[i]

# ─── Centroides en coordenadas de píxel (para marcar los FN) ─
inv_t = ~transform_val
def centroide_px(geom):
    c = geom.centroid
    col, row = inv_t * (c.x, c.y)
    return col, row

fig, axes = plt.subplots(1, 2, figsize=(24, 11),
                         gridspec_kw={'width_ratios': [1.5, 1]})
fig.suptitle(
    f'Aciertos y desaciertos por evento — {NOMBRE_AREA}  ({NOMBRE_EVAL})\n'
    f'TP: {m["TP"]}/{m["n_total"]}  |  FN: {m["FN"]}  |  '
    f'Recall: {m["Recall"]:.3f}  |  Precisión: {m["Precision"]:.3f}  |  F1: {m["F1"]:.3f}',
    fontsize=15, fontweight='bold'
)

# ─── Panel 1: mapa ───────────────────────────────────────────
ax = axes[0]
ax.imshow(ndvi_diff_arr, cmap='Greys', alpha=0.45,
          vmin=np.nanpercentile(ndvi_diff_arr, 2),
          vmax=np.nanpercentile(ndvi_diff_arr, 98))

# Polígonos del inventario: verde si detectado, naranja si no
ov = np.zeros((*mask_inv.shape, 4), dtype=float)
ov[mask_tp] = [0.10, 0.60, 0.15, 0.55]
ov[mask_fn] = [1.00, 0.55, 0.00, 0.75]
ax.imshow(ov)

# Píxeles detectados
ov2 = np.zeros((*mask_inv.shape, 4), dtype=float)
ov2[det_fuera]  = [0.85, 0.05, 0.05, 0.85]   # FP
ov2[det_dentro] = [0.00, 0.35, 0.00, 0.95]   # detección sobre inventario
ax.imshow(ov2)

# Círculo sobre cada FN para que no se pierdan los polígonos pequeños
for i in m['fn_ids']:
    col, row = centroide_px(gdf_inv.geometry.iloc[i])
    ax.plot(col, row, marker='o', markersize=13, markerfacecolor='none',
            markeredgecolor='darkorange', markeredgewidth=2.0, zorder=5)

ax.set_title(f'{np.sum(MASK_EVAL):,} px detectados  |  '
             f'{m["FP_px"]:,} px fuera del inventario',
             fontsize=12, fontweight='bold')
ax.axis('off')
ax.legend(handles=[
    Patch(facecolor='#1a9919', alpha=0.7,  label=f'TP — detectados: {m["TP"]}'),
    Patch(facecolor='darkorange', alpha=0.8, label=f'FN — no detectados: {m["FN"]}'),
    Patch(facecolor='#005900',               label='Píxeles sobre inventario'),
    Patch(facecolor='#d90d0d', alpha=0.9,    label=f'FP — píxeles fuera: {m["FP_px"]:,}'),
    Line2D([0], [0], marker='o', color='darkorange', linestyle='none',
           markerfacecolor='none', markersize=11, label='Ubicación de cada FN'),
], loc='lower right', fontsize=11, framealpha=0.9)

# ─── Panel 2: ¿de qué tamaño son los que se escapan? ─────────
ax = axes[1]
areas = gdf_inv['area_m2'].values
det   = np.zeros(len(gdf_inv), dtype=bool)
det[m['tp_ids']] = True

ax.scatter(np.where(det)[0] * 0 + 1 + np.random.uniform(-0.18, 0.18, det.sum()),
           areas[det], s=55, color='green', alpha=0.75,
           edgecolor='white', linewidth=0.6, label=f'Detectados (n={det.sum()})')
ax.scatter(np.where(~det)[0] * 0 + 2 + np.random.uniform(-0.18, 0.18, (~det).sum()),
           areas[~det], s=55, color='darkorange', alpha=0.85,
           edgecolor='white', linewidth=0.6, label=f'No detectados (n={(~det).sum()})')

for pos, sel, color in [(1, det, 'darkgreen'), (2, ~det, 'saddlebrown')]:
    if sel.sum() > 0:
        med = np.median(areas[sel])
        ax.hlines(med, pos - 0.3, pos + 0.3, color=color, linewidth=2.5)
        ax.text(pos + 0.34, med, f'mediana\n{med:,.0f} m²',
                fontsize=10, color=color, va='center')

ax.set_yscale('log')
ax.set_xlim(0.4, 2.9)
ax.set_xticks([1, 2])
ax.set_xticklabels(['Detectados\n(TP)', 'No detectados\n(FN)'], fontsize=12)
ax.set_ylabel('Área del deslizamiento (m², escala log)', fontsize=12)
ax.set_title('Tamaño de los eventos según detección\n'
             '(si los FN son sistemáticamente pequeños, es límite de resolución)',
             fontsize=12, fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(fontsize=11, loc='upper right')

plt.tight_layout()
nombre_archivo = NOMBRE_AREA.lower().replace(' ', '_')
plt.savefig(f'/tmp/{nombre_archivo}_aciertos_desaciertos.png', dpi=180, bbox_inches='tight')
plt.show()

# ─── Listado de los eventos no detectados ────────────────────
print(f"✓ Guardado en /tmp/{nombre_archivo}_aciertos_desaciertos.png\n")
print(f"Deslizamientos NO detectados ({m['FN']} de {m['n_total']}):")
if m['fn_ids']:
    for i in sorted(m['fn_ids'], key=lambda k: -gdf_inv['area_m2'].iloc[k])[:15]:
        c = gdf_inv.geometry.iloc[i].centroid
        print(f"  id {i:>4}  |  {gdf_inv['area_m2'].iloc[i]:>10,.0f} m²  |  "
              f"lon {c.x:.5f}, lat {c.y:.5f}")
    if m['FN'] > 15:
        print(f"  ... y {m['FN']-15} más")
    print(f"\n  Área mediana FN: {np.median(areas[~det]):,.0f} m²  |  "
          f"TP: {np.median(areas[det]):,.0f} m²")
else:
    print("  Ninguno — recall = 1.000")

In [ ]:
# ============================================================
# 16b - Mapa de aciertos y desaciertos (validación por evento)
# Verde  = deslizamiento detectado (TP)
# Naranja= deslizamiento NO detectado (FN)
# Rojo   = píxeles detectados fuera del inventario (FP)
# Depende de: gdf_inv, masks_individuales, mask_inv, metricas_por_evento,
#             ndvi_diff_arr, transform_val, NOMBRE_AREA
# ============================================================

from matplotlib.patches import Patch

# ─── Máscara a evaluar (cambiar aquí para comparar etapas) ───
MASK_EVAL   = mask_final
NOMBRE_EVAL = 'Pipeline completo'

m = metricas_por_evento(MASK_EVAL, masks_individuales, gdf_inv)

# ─── Máscaras derivadas ──────────────────────────────────────
det_dentro = MASK_EVAL & mask_inv          # píxeles detectados sobre inventario
det_fuera  = MASK_EVAL & ~mask_inv         # falsos positivos espaciales

mask_tp = np.zeros_like(mask_inv)
for i in m['tp_ids']:
    mask_tp |= masks_individuales[i]
mask_fn = np.zeros_like(mask_inv)
for i in m['fn_ids']:
    mask_fn |= masks_individuales[i]

fig, axes = plt.subplots(1, 2, figsize=(24, 11),
                         gridspec_kw={'width_ratios': [1.5, 1]})
fig.suptitle(
    f'Aciertos y desaciertos por evento — {NOMBRE_AREA}  ({NOMBRE_EVAL})\n'
    f'TP: {m["TP"]}/{m["n_total"]}  |  FN: {m["FN"]}  |  '
    f'Recall: {m["Recall"]:.3f}  |  Precisión: {m["Precision"]:.3f}  |  F1: {m["F1"]:.3f}',
    fontsize=15, fontweight='bold'
)

# ─── Panel 1: mapa ───────────────────────────────────────────
ax = axes[0]
ax.imshow(ndvi_diff_arr, cmap='Greys', alpha=0.45,
          vmin=np.nanpercentile(ndvi_diff_arr, 2),
          vmax=np.nanpercentile(ndvi_diff_arr, 98))

# Polígonos del inventario: verde si detectado, naranja si no
ov = np.zeros((*mask_inv.shape, 4), dtype=float)
ov[mask_tp] = [0.10, 0.60, 0.15, 0.55]
ov[mask_fn] = [1.00, 0.55, 0.00, 1.00]
ax.imshow(ov)

# Píxeles detectados
ov2 = np.zeros((*mask_inv.shape, 4), dtype=float)
ov2[det_fuera]  = [0.85, 0.05, 0.05, 0.85]   # FP
ov2[det_dentro] = [0.00, 0.35, 0.00, 0.95]   # detección sobre inventario
ax.imshow(ov2)

ax.set_title(f'{np.sum(MASK_EVAL):,} px detectados  |  '
             f'{m["FP_px"]:,} px fuera del inventario',
             fontsize=12, fontweight='bold')
ax.axis('off')
ax.legend(handles=[
    Patch(facecolor='#1a9919', alpha=0.7,  label=f'TP — detectados: {m["TP"]}'),
    Patch(facecolor='darkorange', alpha=0.8, label=f'FN — no detectados: {m["FN"]}'),
    Patch(facecolor='#005900',               label='Píxeles sobre inventario'),
    Patch(facecolor='#d90d0d', alpha=0.9,    label=f'FP — píxeles fuera: {m["FP_px"]:,}'),
], loc='lower right', fontsize=11, framealpha=0.9)

# ─── Panel 2: ¿de qué tamaño son los que se escapan? ─────────
ax = axes[1]
areas = gdf_inv['area_m2'].values
det   = np.zeros(len(gdf_inv), dtype=bool)
det[m['tp_ids']] = True

ax.scatter(np.where(det)[0] * 0 + 1 + np.random.uniform(-0.18, 0.18, det.sum()),
           areas[det], s=55, color='green', alpha=0.75,
           edgecolor='white', linewidth=0.6, label=f'Detectados (n={det.sum()})')
ax.scatter(np.where(~det)[0] * 0 + 2 + np.random.uniform(-0.18, 0.18, (~det).sum()),
           areas[~det], s=55, color='darkorange', alpha=0.85,
           edgecolor='white', linewidth=0.6, label=f'No detectados (n={(~det).sum()})')

for pos, sel, color in [(1, det, 'darkgreen'), (2, ~det, 'saddlebrown')]:
    if sel.sum() > 0:
        med = np.median(areas[sel])
        ax.hlines(med, pos - 0.3, pos + 0.3, color=color, linewidth=2.5)
        ax.text(pos + 0.34, med, f'mediana\n{med:,.0f} m²',
                fontsize=10, color=color, va='center')

ax.set_yscale('log')
ax.set_xlim(0.4, 2.9)
ax.set_xticks([1, 2])
ax.set_xticklabels(['Detectados\n(TP)', 'No detectados\n(FN)'], fontsize=12)
ax.set_ylabel('Área del deslizamiento (m², escala log)', fontsize=12)
ax.set_title('Tamaño de los eventos según detección\n'
             '(si los FN son sistemáticamente pequeños, es límite de resolución)',
             fontsize=12, fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(fontsize=11, loc='upper right')

plt.tight_layout()
nombre_archivo = NOMBRE_AREA.lower().replace(' ', '_')
plt.savefig(f'/tmp/{nombre_archivo}_aciertos_desaciertos.png', dpi=180, bbox_inches='tight')
plt.show()

# ─── Listado de los eventos no detectados ────────────────────
print(f"✓ Guardado en /tmp/{nombre_archivo}_aciertos_desaciertos.png\n")
print(f"Deslizamientos NO detectados ({m['FN']} de {m['n_total']}):")
if m['fn_ids']:
    for i in sorted(m['fn_ids'], key=lambda k: -gdf_inv['area_m2'].iloc[k])[:15]:
        c = gdf_inv.geometry.iloc[i].centroid
        print(f"  id {i:>4}  |  {gdf_inv['area_m2'].iloc[i]:>10,.0f} m²  |  "
              f"lon {c.x:.5f}, lat {c.y:.5f}")
    if m['FN'] > 15:
        print(f"  ... y {m['FN']-15} más")
    print(f"\n  Área mediana FN: {np.median(areas[~det]):,.0f} m²  |  "
          f"TP: {np.median(areas[det]):,.0f} m²")
else:
    print("  Ninguno — recall = 1.000")

## 14 - Clustering + Widget para validacion

In [ ]:
!pip install scikit-learn

In [ ]:
# ============================================================
# 13 - Clusterización DBSCAN + Widget interactivo de validación
# Dependencias: mask_final (todos los filtros), ndvi_diff_arr, rgb_pre_raw,
#               rgb_post_raw, slope_resampled, gdf_susc,
#               bounds, transform_ndvi, p99_f
# ============================================================

import numpy as np
from sklearn.cluster import DBSCAN
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# ─── 1. DBSCAN sobre píxeles candidatos P_NDVI ─────────────────
print("Ejecutando DBSCAN...")

# Obtener coordenadas de píxeles candidatos
coords_px = np.column_stack(np.where(mask_final))  # usar mask_final (todos los filtros)

# DBSCAN en espacio de píxeles (1 píxel = 10m)
db = DBSCAN(eps=2, min_samples=5, metric='euclidean', n_jobs=-1).fit(coords_px)
labels = db.labels_

n_clusters  = len(set(labels)) - (1 if -1 in labels else 0)
n_noise     = np.sum(labels == -1)
print(f"✓ Clusters encontrados: {n_clusters}")
print(f"  Píxeles ruido descartados: {n_noise:,}")

# Mapa de etiquetas sobre el array completo
label_map = np.full(mask_final.shape, -1, dtype=int)
label_map[mask_final] = labels

# ─── 2. Estadísticos por cluster ────────────────────────────
print("\nCalculando estadísticos por cluster...")

PIXEL_AREA_M2 = 10 * 10  # 10m × 10m
BUFFER_PX     = 5         # 50m / 10m por píxel

lon_min, lat_min, lon_max, lat_max = bounds
h_arr, w_arr = mask_p99_final.shape
px_lon = (lon_max - lon_min) / w_arr  # grados por píxel en X
px_lat = (lat_max - lat_min) / h_arr  # grados por píxel en Y

# Susceptibilidad dominante por píxel (rasterizada)
susc_order  = {'Muy Alto': 5, 'Alto': 4, 'Medio': 3, 'Bajo': 2, 'Muy Bajo': 1}
susc_map_arr = np.zeros(mask_p99_final.shape, dtype=int)
for nivel, orden in susc_order.items():
    subset = gdf_susc[gdf_susc['Nivel_Susc'] == nivel]
    if not subset.empty:
        sub_rast = rasterize(
            [(g, orden) for g in subset.geometry],
            out_shape=mask_p99_final.shape,
            transform=transform_ndvi,
            fill=0, dtype='int32', all_touched=True
        )
        susc_map_arr = np.maximum(susc_map_arr, sub_rast)

susc_inv = {v: k for k, v in susc_order.items()}

records = []
for cid in range(n_clusters):
    mask_c = label_map == cid
    rows, cols = np.where(mask_c)

    n_px       = int(np.sum(mask_c))
    area_m2    = n_px * PIXEL_AREA_M2
    area_ha    = area_m2 / 10000

    # Centroide en coordenadas geográficas
    row_c = int(rows.mean())
    col_c = int(cols.mean())
    lon_c = lon_min + col_c * px_lon
    lat_c = lat_max - row_c * px_lat

    # ΔNDVI
    vals_ndvi  = ndvi_diff_arr[mask_c]
    vals_ndvi  = vals_ndvi[~np.isnan(vals_ndvi)]
    ndvi_mean  = float(np.mean(vals_ndvi))  if len(vals_ndvi) else np.nan
    ndvi_max   = float(np.max(vals_ndvi))   if len(vals_ndvi) else np.nan
    ndvi_p95   = float(np.percentile(vals_ndvi, 95)) if len(vals_ndvi) else np.nan

    # Pendiente media
    vals_slope = slope_resampled[mask_c]
    slope_mean = float(np.nanmean(vals_slope))

    # Susceptibilidad dominante
    vals_susc  = susc_map_arr[mask_c]
    susc_dom   = susc_inv.get(int(np.median(vals_susc[vals_susc > 0])), 'N/D') if np.any(vals_susc > 0) else 'N/D'

    # Bounding box con buffer
    r_min = max(0, int(rows.min()) - BUFFER_PX)
    r_max = min(h_arr, int(rows.max()) + BUFFER_PX)
    c_min = max(0, int(cols.min()) - BUFFER_PX)
    c_max = min(w_arr, int(cols.max()) + BUFFER_PX)

    records.append({
        'id':          cid,
        'n_pixeles':   n_px,
        'area_m2':     area_m2,
        'area_ha':     round(area_ha, 4),
        'lon':         round(lon_c, 6),
        'lat':         round(lat_c, 6),
        'ndvi_mean':   round(ndvi_mean, 4),
        'ndvi_max':    round(ndvi_max,  4),
        'ndvi_p95':    round(ndvi_p95,  4),
        'slope_mean':  round(slope_mean, 1),
        'susc_dom':    susc_dom,
        'r_min': r_min, 'r_max': r_max,
        'c_min': c_min, 'c_max': c_max,
        'validacion':  'Pendiente'
    })

df_clusters = pd.DataFrame(records).sort_values('area_m2', ascending=False).reset_index(drop=True)
print(f"✓ Tabla de {len(df_clusters)} clusters generada")
print(df_clusters[['id','area_ha','ndvi_mean','ndvi_max','slope_mean','susc_dom']].head(10).to_string(index=False))

# ─── 3. Widget interactivo de validación ────────────────────
validaciones = {row['id']: 'Pendiente' for _, row in df_clusters.iterrows()}

def plot_candidato(idx):
    row    = df_clusters.iloc[idx]
    cid    = row['id']
    r_min, r_max = int(row['r_min']), int(row['r_max'])
    c_min, c_max = int(row['c_min']), int(row['c_max'])
    mask_c = (label_map == cid)[r_min:r_max, c_min:c_max]

    # Recortes
    def crop_rgb(arr):
        c = arr[r_min:r_max, c_min:c_max].astype(float)
        if c.max() > 1:
            c = c / 255.0
        # Normalizar localmente
        for ch in range(3):
            p2, p98 = np.nanpercentile(c[..., ch], 2), np.nanpercentile(c[..., ch], 98)
            c[..., ch] = np.clip((c[..., ch] - p2) / (p98 - p2 + 1e-10), 0, 1)
        return c

    rgb_pre_crop  = crop_rgb(rgb_pre_raw)
    rgb_post_crop = crop_rgb(rgb_post_raw)
    ndvi_crop     = ndvi_diff_arr[r_min:r_max, c_min:c_max]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    estado = validaciones.get(cid, 'Pendiente')
    color_titulo = {'Confirmado': 'green', 'Descartado': 'red',
                    'Dudoso': 'orange', 'Pendiente': 'gray'}
    fig.suptitle(
        f'Candidato #{idx+1} / {len(df_clusters)}  —  ID {cid}  '
        f'|  {row["area_ha"]} ha  |  Susc: {row["susc_dom"]}  '
        f'|  Pendiente media: {row["slope_mean"]}°\n'
        f'Lon: {row["lon"]}  Lat: {row["lat"]}  '
        f'|  Estado: {estado}',
        fontsize=11, fontweight='bold',
        color=color_titulo.get(estado, 'gray')
    )

    # Panel 1: RGB Pre
    axes[0].imshow(rgb_pre_crop)
    axes[0].contour(mask_c.astype(int), levels=[0.5], colors='yellow', linewidths=1.5)
    axes[0].set_title('RGB Pre-evento', fontsize=10, fontweight='bold')
    axes[0].axis('off')

    # Panel 2: RGB Post
    axes[1].imshow(rgb_post_crop)
    axes[1].contour(mask_c.astype(int), levels=[0.5], colors='yellow', linewidths=1.5)
    axes[1].set_title('RGB Post-evento', fontsize=10, fontweight='bold')
    axes[1].axis('off')

    # Panel 3: ΔNDVI
    vmax_local = np.nanpercentile(np.abs(ndvi_crop), 98) if np.any(~np.isnan(ndvi_crop)) else 0.3
    im = axes[2].imshow(ndvi_crop, cmap='RdYlGn_r', vmin=-vmax_local, vmax=vmax_local)
    axes[2].contour(mask_c.astype(int), levels=[0.5], colors='black', linewidths=1.5)
    axes[2].set_title(
        f'ΔNDVI  |  media: {row["ndvi_mean"]}  |  máx: {row["ndvi_max"]}',
        fontsize=10, fontweight='bold'
    )
    axes[2].axis('off')
    plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04, label='ΔNDVI')

    plt.tight_layout()
    plt.show()


# ─── 4. Controles del widget ─────────────────────────────────
slider = widgets.IntSlider(
    value=0, min=0, max=len(df_clusters)-1, step=1,
    description='Candidato:',
    layout=widgets.Layout(width='500px'),
    style={'description_width': '100px'}
)

btn_confirmar = widgets.Button(description='✅ Confirmar',
                               button_style='success',
                               layout=widgets.Layout(width='160px'))
btn_descartar = widgets.Button(description='❌ Descartar',
                               button_style='danger',
                               layout=widgets.Layout(width='160px'))
btn_dudoso    = widgets.Button(description='⚠️  Dudoso',
                               button_style='warning',
                               layout=widgets.Layout(width='160px'))
btn_exportar  = widgets.Button(description='💾 Exportar CSV',
                               button_style='info',
                               layout=widgets.Layout(width='160px'))

out_status = widgets.Output()

def on_confirmar(b):
    cid = int(df_clusters.iloc[slider.value]['id'])
    validaciones[cid] = 'Confirmado'
    df_clusters.loc[df_clusters['id'] == cid, 'validacion'] = 'Confirmado'
    with out_status:
        clear_output(wait=True)
        print(f"✅ Candidato ID {cid} marcado como Confirmado")
    # Avanzar al siguiente automáticamente
    if slider.value < len(df_clusters) - 1:
        slider.value += 1

def on_descartar(b):
    cid = int(df_clusters.iloc[slider.value]['id'])
    validaciones[cid] = 'Descartado'
    df_clusters.loc[df_clusters['id'] == cid, 'validacion'] = 'Descartado'
    with out_status:
        clear_output(wait=True)
        print(f"❌ Candidato ID {cid} marcado como Descartado")
    if slider.value < len(df_clusters) - 1:
        slider.value += 1

def on_dudoso(b):
    cid = int(df_clusters.iloc[slider.value]['id'])
    validaciones[cid] = 'Dudoso'
    df_clusters.loc[df_clusters['id'] == cid, 'validacion'] = 'Dudoso'
    with out_status:
        clear_output(wait=True)
        print(f"⚠️  Candidato ID {cid} marcado como Dudoso")
    if slider.value < len(df_clusters) - 1:
        slider.value += 1

def on_exportar(b):
    path_csv = '/tmp/validacion_candidatos.csv'
    df_export = df_clusters[['id','area_ha','lon','lat','ndvi_mean',
                              'ndvi_max','ndvi_p95','slope_mean',
                              'susc_dom','validacion']]
    df_export.to_csv(path_csv, index=False)
    resumen = df_clusters['validacion'].value_counts().to_dict()
    with out_status:
        clear_output(wait=True)
        print(f"💾 CSV exportado en: {path_csv}")
        print(f"   Resumen: {resumen}")

btn_confirmar.on_click(on_confirmar)
btn_descartar.on_click(on_descartar)
btn_dudoso.on_click(on_dudoso)
btn_exportar.on_click(on_exportar)

# Contador de progreso
progreso = widgets.HTML(value='')

def actualizar_progreso():
    total     = len(df_clusters)
    revisados = sum(1 for v in validaciones.values() if v != 'Pendiente')
    conf      = sum(1 for v in validaciones.values() if v == 'Confirmado')
    desc      = sum(1 for v in validaciones.values() if v == 'Descartado')
    dud       = sum(1 for v in validaciones.values() if v == 'Dudoso')
    progreso.value = (
        f"<b>Progreso: {revisados}/{total}</b> &nbsp;|&nbsp; "
        f"✅ {conf} &nbsp; ❌ {desc} &nbsp; ⚠️ {dud} &nbsp; "
        f"⏳ {total - revisados} pendientes"
    )

def on_slider_change(change):
    actualizar_progreso()
    with out_plot:
        clear_output(wait=True)
        plot_candidato(change['new'])

slider.observe(on_slider_change, names='value')

out_plot = widgets.Output()

# Layout final
controles = widgets.HBox([btn_confirmar, btn_descartar, btn_dudoso, btn_exportar])
ui = widgets.VBox([
    progreso,
    slider,
    controles,
    out_status,
    out_plot
])

actualizar_progreso()
display(ui)

# Mostrar primer candidato
with out_plot:
    plot_candidato(0)

## 15 - Filtro de compacidad

In [ ]:
# ============================================================
# 13d - Filtro de Compacidad sobre clusters DBSCAN
# ============================================================

from shapely.geometry import shape as shapely_shape
from rasterio.features import shapes
from rasterio.transform import from_bounds
import matplotlib.cm as cm
from matplotlib.colors import Normalize as MplNorm
from matplotlib.patches import Patch

lon_min_aoi, lat_min_aoi, lon_max_aoi, lat_max_aoi = bounds
h_arr, w_arr = mask_final.shape

transform_comp = from_bounds(
    lon_min_aoi, lat_min_aoi,
    lon_max_aoi, lat_max_aoi,
    w_arr, h_arr
)

# ─── 1. Calcular compacidad por cluster ─────────────────────
print("Calculando compacidad de clusters...")

compacidades = []
for _, row in df_clusters.iterrows():
    cid_0  = row['id']
    mask_c = (label_map == cid_0).astype(np.uint8)
    polys  = [shapely_shape(s) for s, v in shapes(mask_c, transform=transform_comp) if v == 1]
    if not polys:
        compacidades.append(np.nan)
        continue
    geom_c  = polys[0]
    compac  = (4 * np.pi * geom_c.area) / (geom_c.length ** 2 + 1e-10)
    compacidades.append(round(compac, 4))

df_clusters['compacidad'] = compacidades

# ─── 2. Clasificar por forma ─────────────────────────────────
COMPAC_UMBRAL = 0.75

def clasificar_forma(c):
    if np.isnan(c):          return 'Sin datos'
    elif c <= 0.30:          return 'Elongado'
    elif c <= 0.55:          return 'Moderado'
    elif c <= COMPAC_UMBRAL: return 'Compacto'
    else:                    return 'Muy circular ⚠'

df_clusters['forma'] = df_clusters['compacidad'].apply(clasificar_forma)

print(f"  Distribución por forma:")
for forma, cnt in df_clusters['forma'].value_counts().items():
    print(f"    {forma:20s}: {cnt:3d} ({100*cnt/len(df_clusters):.1f}%)")

# ─── 3. Construir mapa de colores en UN solo array ───────────
# En vez de 150 imshow, un único array RGBA
print("\nGenerando mapa de compacidad...")

cmap_comp = cm.RdYlGn_r
norm_comp  = MplNorm(vmin=0, vmax=1)

# Array RGBA único para todos los clusters
color_map_rgba = np.zeros((h_arr, w_arr, 4), dtype=float)

for _, row in df_clusters.iterrows():
    cid_0  = row['id']
    c_val  = row['compacidad']
    if np.isnan(c_val):
        continue
    mask_c = label_map == cid_0
    color  = cmap_comp(norm_comp(c_val))
    alpha  = 0.95 if c_val > COMPAC_UMBRAL else 0.80
    color_map_rgba[mask_c] = [color[0], color[1], color[2], alpha]

# ─── 4. Visualización ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.suptitle('Filtro de Compacidad — Forma de los clusters\Àrea de estudio',
             fontsize=14, fontweight='bold')

# Panel 1: histograma
ax = axes[0]
vals_comp = df_clusters['compacidad'].dropna()
ax.hist(vals_comp, bins=30, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(COMPAC_UMBRAL, color='red', linewidth=2.5, linestyle='--',
           label=f'Umbral sospechoso = {COMPAC_UMBRAL}')
ax.axvline(0.30, color='green',  linewidth=1.5, linestyle=':', label='Elongado < 0.30')
ax.axvline(0.55, color='orange', linewidth=1.5, linestyle=':', label='Moderado < 0.55')

for x0, x1, label, color in [
    (0,    0.30, f'Elongado\n{(vals_comp<=0.30).sum()}',                         'green'),
    (0.30, 0.55, f'Moderado\n{((vals_comp>0.30)&(vals_comp<=0.55)).sum()}',      'goldenrod'),
    (0.55, 0.75, f'Compacto\n{((vals_comp>0.55)&(vals_comp<=0.75)).sum()}',      'orange'),
    (0.75, 1.00, f'Muy circular\n{(vals_comp>0.75).sum()} ⚠',                   'red'),
]:
    ax.axvspan(x0, x1, alpha=0.08, color=color)

ax.set_xlabel('Compacidad', fontsize=12)
ax.set_ylabel('Número de clusters', fontsize=12)
ax.set_title('Distribución de compacidad', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.spines[['top','right']].set_visible(False)

# Panel 2: mapa — UN solo imshow en vez de 150
ax = axes[1]
ax.imshow(slope_resampled, cmap='terrain',
          vmin=0, vmax=np.nanpercentile(slope_resampled, 99), alpha=0.5)
ax.imshow(color_map_rgba)  # ← único imshow con todos los clusters

sm = cm.ScalarMappable(cmap=cmap_comp, norm=norm_comp)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.035, pad=0.03, shrink=0.7)
cbar.set_label('Compacidad', fontsize=11)
cbar.ax.axhline(COMPAC_UMBRAL, color='red', linewidth=2, linestyle='--')
cbar.ax.text(1.1, COMPAC_UMBRAL, f'⚠ {COMPAC_UMBRAL}', va='center',
             fontsize=9, color='red', transform=cbar.ax.transAxes)

ax.set_title('Compacidad por cluster\n(verde=elongado ✓  |  rojo=circular ⚠)',
             fontsize=12, fontweight='bold')
ax.axis('off')
ax.legend(handles=[
    Patch(facecolor='green',  alpha=0.8, label='Elongado (≤ 0.30)'),
    Patch(facecolor='gold',   alpha=0.8, label='Moderado (0.30–0.55)'),
    Patch(facecolor='orange', alpha=0.8, label='Compacto (0.55–0.75)'),
    Patch(facecolor='red',    alpha=0.9, label=f'Muy circular > {COMPAC_UMBRAL} ⚠'),
], loc='lower right', fontsize=9, framealpha=0.9)

plt.tight_layout()
plt.savefig('/tmp/venecia_compacidad.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Guardado en /tmp/venecia_compacidad.png")
print(f"\ndf_clusters actualizado con columnas 'compacidad' y 'forma'")
print("Agrega row['forma'] en el widget para verlo en la validación.")

## 16 Exportar shapefile Inventario pre-seleccionado

In [ ]:
# ============================================================
# 14 - Exportar Shapefiles de candidatos y confirmados
# ============================================================

import geopandas as gpd
from shapely.geometry import MultiPolygon
from rasterio.features import shapes
from rasterio.transform import from_bounds
from shapely.geometry import shape as shapely_shape
import numpy as np, os
from datetime import datetime

ts         = datetime.now().strftime('%Y%m%d_%H%M%S')
output_dir = "/home/ecastilloc/Descargas/WP.1.2.5/4_Febrero2026/resultados"
os.makedirs(output_dir, exist_ok=True)

lon_min_aoi, lat_min_aoi, lon_max_aoi, lat_max_aoi = bounds
h_arr, w_arr = mask_p99_final.shape

transform_export = from_bounds(
    lon_min_aoi, lat_min_aoi,
    lon_max_aoi, lat_max_aoi,
    w_arr, h_arr
)

def cluster_to_polygon(cid_0based):
    mask_c = (label_map == cid_0based).astype(np.uint8)
    polys  = [shapely_shape(s) for s, v in shapes(mask_c, transform=transform_export) if v == 1]
    if not polys:
        return None
    return polys[0] if len(polys) == 1 else MultiPolygon(polys)

# ─── Vectorizar todos los clusters ──────────────────────────
print("Vectorizando clusters...")
geometrias = []
for _, row in df_clusters.iterrows():
    geom = cluster_to_polygon(row['id'])   # ← 'id' en minúscula
    geometrias.append(geom)

# ─── 1. Shapefile — TODOS los candidatos ────────────────────
df_export = df_clusters.drop(columns=['r_min','r_max','c_min','c_max']).copy()

gdf_todos = gpd.GeoDataFrame(df_export, geometry=geometrias, crs='EPSG:4326')
gdf_todos = gdf_todos[gdf_todos.geometry.notna()]

path_todos = os.path.join(output_dir, f'candidatos_todos_{ts}.shp')
gdf_todos.to_file(path_todos)
print(f"✓ Todos los candidatos: {path_todos}  ({len(gdf_todos)} polígonos)")

# ─── 2. Shapefile — Solo CONFIRMADOS ────────────────────────
gdf_conf = gdf_todos[gdf_todos['validacion'] == 'Confirmado'].copy()

if len(gdf_conf) > 0:
    path_conf = os.path.join(output_dir, f'deslizamientos_confirmados_{ts}.shp')
    gdf_conf.to_file(path_conf)
    print(f"✓ Confirmados:         {path_conf}  ({len(gdf_conf)} polígonos)")
else:
    print("⚠ No hay candidatos confirmados aún — valida primero en el widget")

# ─── 3. Resumen ──────────────────────────────────────────────
print(f"\nResumen de validación:")
for estado, count in df_clusters['validacion'].value_counts().items():
    print(f"  {estado:12s}: {count:3d} candidatos")

print(f"\nÁrea total confirmada: {gdf_conf['area_ha'].sum():.4f} ha" if len(gdf_conf) > 0 else "")

In [ ]:
# ============================================================
# 15 - Resumen de áreas: municipio, candidatos y confirmados
# ============================================================

from pyproj import Geod

geod = Geod(ellps="WGS84")

# ─── 1. Área del municipio (shapefile original) ──────────────
area_mun_m2  = abs(geod.geometry_area_perimeter(gdf.geometry.iloc[0])[0])
area_mun_ha  = area_mun_m2 / 10_000
area_mun_km2 = area_mun_m2 / 1_000_000

# ─── 2. Área total de candidatos (todos los clusters) ────────
area_cand_ha  = df_clusters['area_ha'].sum()
area_cand_km2 = area_cand_ha / 100
area_cand_m2  = area_cand_ha * 10_000
pct_cand      = (area_cand_ha / area_mun_ha) * 100

# ─── 3. Área de deslizamientos confirmados ───────────────────
df_conf       = df_clusters[df_clusters['validacion'] == 'Confirmado']
area_conf_ha  = df_conf['area_ha'].sum()
area_conf_km2 = area_conf_ha / 100
area_conf_m2  = area_conf_ha * 10_000
pct_conf_mun  = (area_conf_ha / area_mun_ha) * 100
pct_conf_cand = (area_conf_ha / area_cand_ha * 100) if area_cand_ha > 0 else 0

# ─── 4. Imprimir resumen ─────────────────────────────────────
sep = "─" * 52

print(sep)
print(f"  RESUMEN DE ÁREAS — {gdf['NOMBRE'].iloc[0] if 'NOMBRE' in gdf.columns else 'Municipio'}")
print(sep)

print(f"\n📍 Área de estudio (municipio):")
print(f"   {area_mun_ha:>12,.2f}  ha")
print(f"   {area_mun_km2:>12,.2f}  km²")

print(f"\n🟠 Candidatos a deslizamiento ({LBL_NDVI} + pendiente + susceptibilidad):")
print(f"   {area_cand_ha:>12,.4f}  ha")
print(f"   {area_cand_km2:>12,.4f}  km²")
print(f"   {pct_cand:>11,.3f}%  del municipio")
print(f"   {len(df_clusters):>12d}  clusters")

print(f"\n✅ Deslizamientos confirmados (validación experto):")
print(f"   {area_conf_ha:>12,.4f}  ha")
print(f"   {area_conf_km2:>12,.4f}  km²")
print(f"   {pct_conf_mun:>11,.3f}%  del municipio")
print(f"   {pct_conf_cand:>11,.1f}%  de los candidatos")
print(f"   {len(df_conf):>12d}  polígonos confirmados")

print(f"\n{sep}")

## 17 - Exportar capas y abrir el explorador de filtros
Corre estas dos celdas después de la validación (sección 13).

In [ ]:
# ============================================================
# 17 - Exportar capas para el explorador de filtros (HTML local)
# Correr al FINAL del notebook (después de la celda de validación).
# Genera un .json que se abre en explorador_filtros_ndvi.html
# (botón "Abrir capas" o arrastrando el archivo a la página).
# ============================================================

import os, json, base64
import numpy as np
from rasterio.features import rasterize

# ─── Configuración ──────────────────────────────────────────
EXPORT_DIR   = "/home/ecastilloc/Descargas/WP.1.2.5/4_Febrero2026/resultados"
EXPORT_NAME  = "capas_explorador_Cisneros.json"
NOMBRE_SITIO = "Cisneros – Santo Domingo 2021"
CURV_PCT     = 85                      # percentil de curvatura usado en la celda 10
DBSCAN_EPS, DBSCAN_MIN = 2, 5          # parámetros de la celda 14
NIVELES_SUSC = ['Muy Alto', 'Alto', 'Medio', 'Bajo', 'Muy Bajo']   # bit 0..4

def _enc(arr):
    """Array → dict {dtype, shape, b64}. Conserva float32/float64 para
    que los umbrales en el HTML reproduzcan exactamente los del notebook."""
    arr = np.asarray(arr)
    if arr.dtype == bool:
        arr = arr.astype(np.uint8)
    elif arr.dtype.kind == 'f' and arr.dtype not in (np.float32, np.float64):
        arr = arr.astype(np.float32)
    arr = np.ascontiguousarray(arr)
    return {"dtype": str(arr.dtype), "shape": list(arr.shape),
            "b64": base64.b64encode(arr.tobytes()).decode("ascii")}

h, w = ndvi_diff_arr.shape

# ─── DEM para el hillshade del explorador ───────────────────
# Recorte del DEM 12 m de Antioquia al AOI, reproyectado (bilineal)
# directamente al grid de ΔNDVI: mismo CRS, extensión y tamaño.
# Solo lee los bloques del AOI, no el raster completo.
from rasterio.vrt import WarpedVRT
from rasterio.enums import Resampling

DEM_ANTIOQUIA = "/home/ecastilloc/Descargas/WP.1.2.5/4_Febrero2026/data/dem/Antioquia/Antioquia.tif"

with rasterio.open(DEM_ANTIOQUIA) as src_dem:
    print(f"DEM Antioquia: {src_dem.crs}  |  {src_dem.res[0]:.1f} m  |  {src_dem.width}×{src_dem.height}")
    with WarpedVRT(src_dem, crs="EPSG:4326", transform=transform_ndvi,
                   width=w, height=h, resampling=Resampling.bilinear,
                   src_nodata=src_dem.nodata, nodata=np.nan, dtype="float32") as vrt:
        dem_grid = vrt.read(1).astype(np.float32)

dem_grid[dem_grid < -1000] = np.nan            # nodata residual
n_nan = int(np.isnan(dem_grid).sum())
print(f"✓ DEM recortado al grid ΔNDVI {dem_grid.shape}  |  "
      f"{np.nanmin(dem_grid):.0f}–{np.nanmax(dem_grid):.0f} m  |  sin dato: {n_nan:,} px")
if n_nan > 0.05 * dem_grid.size:
    print("  ⚠ Más del 5 % sin dato: revisa que el AOI esté dentro del DEM de Antioquia")

# ─── Susceptibilidad: un bit por nivel (all_touched, igual que celda 12) ─
susc_bits = np.zeros((h, w), dtype=np.uint8)
niveles_presentes = []
for b, nivel in enumerate(NIVELES_SUSC):
    sub = gdf_susc[gdf_susc['Nivel_Susc'] == nivel]
    if len(sub):
        niveles_presentes.append(nivel)
        r = rasterize([(g, 1) for g in sub.geometry], out_shape=(h, w),
                      transform=transform_ndvi, fill=0, dtype='uint8',
                      all_touched=True)
        susc_bits |= (r.astype(np.uint8) << b)

# ─── Inventario: índices planos de píxeles por evento ───────
idx_list    = [np.flatnonzero(m).astype(np.int32) for m in masks_individuales]
inv_offsets = np.cumsum([0] + [len(i) for i in idx_list]).astype(np.int32)
inv_idx     = np.concatenate(idx_list).astype(np.int32)

# ─── RGB para fondo (uint8) ─────────────────────────────────
rgb_pre_u8  = (np.clip(rgb_pre_disp,  0, 1) * 255).astype(np.uint8)
rgb_post_u8 = (np.clip(rgb_post_disp, 0, 1) * 255).astype(np.uint8)

# ─── Referencia del notebook (para verificar el HTML) ──────
def _ref(mask):
    m = metricas_por_evento(mask, masks_individuales, gdf_inv)
    return {"px": int(np.sum(mask)), "TP": m["TP"], "F1": m["F1"]}

referencia = {
    "ndvi_p90": _ref(masks_ref[90]),
    "ndvi":     _ref(mask_ndvi),
    "bsi":      _ref(mask_p99_bsi),
    "slope":    _ref(mask_p99_bsi & ladera_mask),
    "curv":     _ref(mask_p99_bsi_slope_curv),
    "susc":     _ref(mask_final),
}

curv_desde_dem = any(os.path.exists(dp) for dp in dem_candidatos)

# ─── Capas vectoriales (OSM) y localización ─────────────────
import glob
import geopandas as gpd
from shapely.geometry import box as shp_box

OSM_DIR         = "/home/ecastilloc/Descargas/WP.1.2.5/8_Junio-Julio-Agosto_2026/2026-2/01_Informacion_base/Informacion_base"
WATER_PATH      = f"{OSM_DIR}/gis_osm_water_a_free_1/gis_osm_water_a_free_1.shp"
WATERWAYS_PATH  = f"{OSM_DIR}/gis_osm_waterways_free_1/gis_osm_waterways_free_1.shp"   # opcional (ríos/quebradas como líneas)
BUILDINGS_DIR   = f"{OSM_DIR}/gis_osm_buildings_a_free_1"
MUNICIPIOS_PATH = "/home/ecastilloc/Descargas/WP.1.2.5/4_Febrero2026/data/municipios/HISTÓRICO_DE_EVENTOS_DAGRAN_-8166032715199014854/municipios_.shp"
COL_MPIO, MUNICIPIO = "MPIO_NOMBR", "SANTO DOMINGO"

aoi_box = shp_box(*bounds)

def _leer_aoi(path):
    """Lee solo lo que toca el AOI (bbox) y lo recorta. None si no existe."""
    if not path or not os.path.exists(path):
        return None
    g = gpd.read_file(path, bbox=gpd.GeoSeries([aoi_box], crs=4326))
    if g.empty:
        return g
    return gpd.clip(g.to_crs(4326), aoi_box)

def _polys(geoms, nd=6):
    """Polígonos → [[anillo plano lon,lat,...], ...] por polígono."""
    out = []
    for g in geoms:
        if g is None or g.is_empty: continue
        for p in getattr(g, "geoms", [g]):
            if p.geom_type != "Polygon": continue
            out.append([np.round(np.asarray(r.coords)[:, :2], nd).ravel().tolist()
                        for r in [p.exterior, *p.interiors]])
    return out

def _lines(geoms, nd=6):
    out = []
    for g in geoms:
        if g is None or g.is_empty: continue
        for l in getattr(g, "geoms", [g]):
            if l.geom_type == "LineString":
                out.append(np.round(np.asarray(l.coords)[:, :2], nd).ravel().tolist())
    return out

bld_shp = (glob.glob(os.path.join(BUILDINGS_DIR, "*.shp")) or [None])[0]
gdf_water = _leer_aoi(WATER_PATH)
gdf_ww    = _leer_aoi(WATERWAYS_PATH)
gdf_bld   = _leer_aoi(bld_shp)

vector = {
    "water":     _polys(gdf_water.geometry) if gdf_water is not None else [],
    "waterways": _lines(gdf_ww.geometry)    if gdf_ww    is not None else [],
    "buildings": _polys(gdf_bld.geometry)   if gdf_bld   is not None else [],
}
print(f"✓ OSM en el AOI: {len(vector['water'])} polígonos de agua, "
      f"{len(vector['waterways'])} drenajes (líneas), {len(vector['buildings']):,} edificios")
if gdf_ww is None:
    print("  (no se encontró gis_osm_waterways_free_1; solo se exportan cuerpos de agua como polígonos)")

# Localización: Antioquia, municipios y municipio de interés (simplificados)
gdf_mun = gpd.read_file(MUNICIPIOS_PATH).to_crs(4326)
gdf_mun[COL_MPIO] = gdf_mun[COL_MPIO].astype(str).str.strip().str.upper()
sel = gdf_mun[gdf_mun[COL_MPIO] == MUNICIPIO.upper()]
if sel.empty:
    raise ValueError(f"No encontré '{MUNICIPIO}' en {COL_MPIO}. Ejemplos: {sorted(gdf_mun[COL_MPIO])[:10]}")

loc = {
    "antioquia":  _polys([gdf_mun.union_all().simplify(0.004)], 4) if hasattr(gdf_mun, "union_all")
                  else _polys([gdf_mun.unary_union.simplify(0.004)], 4),
    "municipios": _polys(gdf_mun.geometry.simplify(0.0015), 4),
    "mpio": {"nombre": MUNICIPIO.upper(), "polys": _polys(sel.geometry.simplify(0.0002), 5)},
}
print(f"✓ Localización: {len(gdf_mun)} municipios  |  {MUNICIPIO} encontrado")

payload = {
    "meta": {
        "sitio": NOMBRE_SITIO,
        "shape": [h, w],
        "bounds": [float(b) for b in bounds],
        "pixel_m": SCALE,
        "pre":  [start_date1, end_date1],
        "post": [start_date2, end_date2],
        "area_min_m2": AREA_MIN_M2,
        "niveles_susc": NIVELES_SUSC,
        "niveles_presentes": niveles_presentes,
        "curv_desde_dem": bool(curv_desde_dem),
        "dem_fuente": "DEM 12 m Antioquia, bilineal al grid de 10 m",
        "defaults": {
            "pct_ndvi": PCT_NDVI, "pct_bsi": PCT_BSI,
            "slope": float(SLOPE_THRESHOLD), "curv_pct": CURV_PCT,
            "pct_ndvi_final": PCT_NDVI_FINAL,
            "susc": ['Muy Alto', 'Alto', 'Medio'],
            "dbscan_eps": DBSCAN_EPS, "dbscan_min": DBSCAN_MIN,
        },
        "ev_area_m2": [round(float(a), 1) for a in gdf_inv['area_m2']],
        "referencia": referencia,
        "vector": vector,
        "loc": loc,
    },
    "layers": {
        "ndvi":      _enc(ndvi_diff_arr),
        "bsi":       _enc(bsi_diff_arr),
        "slope":     _enc(slope_resampled),
        "curv":      _enc(curvatura_arr),
        "dem":       _enc(dem_grid),
        "susc_bits": _enc(susc_bits),
        "rgb_pre":   _enc(rgb_pre_u8),
        "rgb_post":  _enc(rgb_post_u8),
        "inv_idx":   _enc(inv_idx),
        "inv_off":   _enc(inv_offsets),
    },
}

os.makedirs(EXPORT_DIR, exist_ok=True)
out_path = os.path.join(EXPORT_DIR, EXPORT_NAME)
with open(out_path, "w") as f:
    json.dump(payload, f)

print(f"✓ Capas exportadas: {out_path}  ({os.path.getsize(out_path)/1e6:.1f} MB)")
print(f"  Grid {h}×{w}  |  {len(idx_list)} eventos  |  susc: {niveles_presentes}")
print(f"  Curvatura desde DEM: {curv_desde_dem}")


In [ ]:
# ============================================================
# 18 - Abrir el explorador en el navegador
# Pon explorador_filtros_ndvi.html en esta ruta (o cambia HTML_PATH)
# ============================================================
import webbrowser, os
HTML_PATH = "/home/ecastilloc/Descargas/WP.1.2.5/explorador_filtros_ndvi.html"

if os.path.exists(HTML_PATH):
    webbrowser.open("file://" + HTML_PATH)
    print(f"✓ Abierto. Arrastra a la página: {out_path}")
else:
    print(f"⚠ No encontré {HTML_PATH}. Copia ahí el HTML o corrige la ruta.")


In [ ]:
# ============================================================
# 19 - Explorador para compartir (un solo .html con los datos adentro)
# La otra persona solo le da doble clic: no necesita Python,
# el notebook ni el .json. Funciona sin internet.
# ============================================================
import os

HTML_PATH  = "/home/ecastilloc/Descargas/WP.1.2.5/explorador_filtros_ndvi.html"   # plantilla
SHARE_PATH = os.path.join(EXPORT_DIR, "explorador_Cisneros_compartir.html")       # archivo a enviar

with open(HTML_PATH, encoding="utf-8") as f:
    html = f.read()
with open(out_path, encoding="utf-8") as f:          # .json de la celda 17
    capas = f.read().replace("</", "<\\/")           # evita cortar el <script>

marca = "<!--CAPAS_EMBEBIDAS-->"
if marca not in html:
    raise ValueError("La plantilla HTML es una versión vieja: descarga la nueva.")

html = html.replace(marca, "<script>window.CAPAS = " + capas + ";</script>")
with open(SHARE_PATH, "w", encoding="utf-8") as f:
    f.write(html)

print(f"✓ Listo para compartir: {SHARE_PATH}  ({os.path.getsize(SHARE_PATH)/1e6:.1f} MB)")
print("  Súbelo a OneDrive/Drive o envíalo; se abre con doble clic en Chrome, Firefox o Edge.")
